In [0]:
from LDCDataAccessLayerPy import KeyVaultManager, SharePointManager, SqlManager, databricks_init
from datetime import datetime, timedelta
from LDCDataAccessLayerPy import databricks_init, DataLakeManagerGen2
from io import BytesIO
import LDCDataAccessLayerPy
#Initiate the secret to access KeyVault secrets
databricks_init(dbutils, 'GO')
sp_mgr = SharePointManager()
sql_mgr = SqlManager()

import logging
logger = spark._jvm.org.apache.log4j
logging.getLogger("py4j").setLevel(logging.ERROR)

import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.colors import ListedColormap
from sklearn.cluster import KMeans
from openpyxl import load_workbook

from datetime import datetime, timedelta
import re
from dateutil.relativedelta import relativedelta
from LDCDataAccessLayerPy import PriceManager, graph
from LDCDataAccessLayerPy import ZemaManager
import math
import plotly.express as px
import plotly.graph_objects as go
zema = ZemaManager()


In [0]:
start_date = datetime(2018, 1, 1)

end_date = datetime(2030, 1, 1)

In [0]:
url = "https://ldcom365.sharepoint.com"

# EURUSD

In [0]:
eurusd= zema.get_curve(curve='FX_GPL_FWD_EURUSD', period=f"{start_date}::{end_date}")

In [0]:
matba= zema.get_curve(curve='P-FUTURE-MATBA-INPUT-WHEAT-ROS-USD-MT', period=f"{start_date}::{end_date}")

In [0]:
import calendar

def recreate_full_date(contract_month, contract_year):
    # Define the base date and base number
    base_date = datetime(2024, 12, 2)
    base_number = 4336

    # Calculate the number of days offset
    delta_days = contract_month - base_number
    calculated_date = base_date + timedelta(days=delta_days)

    # Extract the day of the month and month number
    day_of_month = calculated_date.day
    month_number = calculated_date.month

    # Get the last day of the target month
    last_day_of_month = calendar.monthrange(contract_year, month_number)[1]

    # Adjust the day if it exceeds the last day of the month
    if day_of_month > last_day_of_month:
        day_of_month = last_day_of_month

    # Use the provided year from the contract_year column
    return datetime(contract_year, month_number, day_of_month)
  

# Define a base date for the mapping
base_date = datetime(2024, 12, 2)
base_number = 4336

# Apply the mapping function to the column
eurusd['Contract Month Date'] = eurusd.apply(
    lambda row: recreate_full_date(row['contract_month'], row['contract_year']), axis=1
)

#eurusd=eurusd[['date','value','contract_year','Contract Month Date']]
eurusd['contract_month']=eurusd['Contract Month Date'].dt.month
eurusd.rename(columns={'value': 'eurusd'}, inplace=True)

# Group by the required columns and calculate the average eurusd
eurusd = eurusd.groupby(['date', 'contract_year', 'contract_month'], as_index=False)['eurusd'].mean()

sp_mgr.save_pd_to_excel('/sites/grainsargprojects/models/zema_prices/FX/EURUSD.xlsx',eurusd,index=False)

# WHEAT

In [0]:
arg = zema.get_curve(curve="P-CASH-LDC-INPUT-FLAT-WHEAT-AR-FOB Up River-11.5-USD-MT", period=f"{start_date}::{end_date}")
matif= zema.get_curve(curve="P-FUTURE-ENXT-INPUT-WHEAT-EUR-MT", period=f"{start_date}::{end_date}")



## MATIF

In [0]:
matif=matif[matif['observation']=='Settle']
matif.rename(columns={'value': 'matif_eur'}, inplace=True)
matif=matif[['date','matif_eur','contract_year','contract_month']]



### MATIF TO USD

In [0]:
merged_df = pd.merge(
    matif,
    eurusd,
    on=['date', 'contract_year', 'contract_month'],
    how='inner'  # You can also use 'outer', 'left', or 'right' depending on your needs
)
merged_df['matif_usd']=merged_df['matif_eur']*merged_df['eurusd']
merged_df=merged_df[['date','matif_usd','contract_year','contract_month']]


In [0]:
matif_raw=merged_df.copy()

## ARG UPR

In [0]:
arg=arg[arg['observation']=='Last']
arg=arg[['date','value','contract_year','contract_month']]
arg['day']=arg['date'].dt.day
arg['month']=arg['date'].dt.month
arg['year']=arg['date'].dt.year
arg['virtual_date'] = pd.to_datetime(
    {'year': 2000, 'month': arg['month'], 'day': arg['day']},
    errors='coerce'  # This will convert invalid dates (e.g. Feb 30) to NaT
)

# Drop rows with invalid virtual dates
arg = arg.dropna(subset=['virtual_date'])

# You can now sort or use this date for seasonal charts
arg = arg.sort_values(by='virtual_date')
arg=arg[['date','value','contract_year','contract_month','virtual_date']]


In [0]:
arg_dec=arg[arg['contract_month']==12]
arg_dec['year']=arg_dec['date'].dt.year


In [0]:
def assign_season_wheat(row):
    date = row['date']
    
    # Check if the date is from Nov to Oct (for the season)
    if date.month >= 11:  # Nov and Dec belong to the current season
        season_start = date.year
    else:  # Jan to Oct belong to the previous season
        season_start = date.year -1 
    
    # Define the season in 'YYYY/YYYY' format
    season = f"{season_start}/{season_start + 1}"
    return season
  

arg_dec['Season'] = arg_dec.apply(assign_season_wheat, axis=1)
arg_dec

In [0]:
arg_dec['virtual_date'] = pd.to_datetime(arg_dec['virtual_date'])

def adjust_virtual_date(date):
    if date.month in [11, 12]:
        return date - pd.DateOffset(years=1) 
    else:
        return date   

# Apply the function to adjust the 'virtual_date'

arg_dec['virtual_date'] = arg_dec['virtual_date'].apply(adjust_virtual_date)


In [0]:
arg_dec = arg_dec.sort_values("virtual_date")



In [0]:
arg_dec = arg_dec.sort_values("virtual_date")

arg_upr_dec = px.line(
    arg_dec,
    x='virtual_date',
    y='value',
    color='Season',
    labels={
        'virtual_date': '',
        'value': 'ARG WHEAT UPR DEC',
        'Season': 'Marketing Year'
    },
    title='UPR WHEAT DEC'
)

arg_upr_dec.update_traces(hovertemplate='%{x|%d-%b}<br>%{y:.2f}<extra>%{fullData.name}</extra>')
arg_upr_dec.update_layout(
    xaxis=dict(
        tickformat='%b',  # Month format like Jan, Feb...
        dtick="M1",
        hoverformat='%d-%b'
    ),
    yaxis_title='value',
    template='plotly_white',
    height=700,
    width=1800,
    hovermode='x unified'
)

arg_upr_dec.show()

arg_upr_dec_html = arg_upr_dec.to_html(include_plotlyjs='cdn', full_html=True)


In [0]:
merged_df = pd.merge(
    arg,
    merged_df,
    on=['date', 'contract_year', 'contract_month'],
    how='inner'  # You can also use 'outer', 'left', or 'right' depending on your needs
)

In [0]:
merged_df['Basis']=merged_df['value']-merged_df['matif_usd']

### CHARTS

In [0]:
mar_contracts = merged_df[merged_df['contract_month'] == 3].copy()

In [0]:
mar_contracts = merged_df[merged_df['contract_month'] == 3].copy()


# Convert to datetime if not already
mar_contracts['virtual_date'] = pd.to_datetime(mar_contracts['virtual_date'])

# Define mask: virtual_date in Jan, Feb, or Mar
mask_q1 = mar_contracts['virtual_date'].dt.month.isin([1, 2, 3])

# Shift those dates by 1 year
mar_contracts.loc[mask_q1, 'virtual_date'] = mar_contracts.loc[mask_q1, 'virtual_date'] + pd.DateOffset(years=1)


pivot = mar_contracts.pivot_table(
    index='virtual_date',
    columns='contract_year',
    values='Basis',
    aggfunc='mean'  # or 'last', 'first', 'max', etc., depending on what makes sense
)

# Step 1: Create a complete weekday date range for virtual year
full_virtual_range = pd.date_range(start="2000-04-01", end="2001-03-31", freq="B")  # 'B' = business day

# Step 2: Reindex the pivot table to this full range
pivot = pivot.reindex(full_virtual_range)

# Ensure all columns are numeric
pivot = pivot.apply(pd.to_numeric, errors='coerce')

# Optional Step 3: Interpolate to fill the gaps (linear or other methods)
pivot_interp = pivot.interpolate(method='linear', limit_direction='both')
pivot_interp = pivot_interp.drop(columns=2020, errors='ignore')


# Drop 2020 from the interpolated pivot
pivot_plotly = pivot_interp.drop(columns=2020, errors='ignore')
cutoff_date = pd.Timestamp("2000-08-04")

# Mask pre-cutoff values for contract year 2021
if 2021 in pivot_plotly.columns:
    pivot_plotly.loc[pivot_plotly.index < cutoff_date, 2021] = np.nan

# Compute average, min, max
mean_curve = pivot_plotly.mean(axis=1)


# Create figure
w_mar = go.Figure()

# Add each contract year as a trace
for year in pivot_plotly.columns:
    w_mar.add_trace(go.Scatter(
        x=pivot_plotly.index,
        y=pivot_plotly[year],
        mode='lines',
        name=str(year),
        line=dict(width=1),
        opacity=0.8,
        hovertemplate='%{x|%d-%b}<br>%{y:.2f}<extra>%{fullData.name}</extra>'

    ))

# Add average curve
w_mar.add_trace(go.Scatter(
    x=pivot_plotly.index,
    y=mean_curve,
    mode='lines',
    name='Average',
    line=dict(color='black', width=2.5),
    hovertemplate='%{x|%d-%b}<br>%{y:.2f}<extra>%{fullData.name}</extra>'

))


# Update layout
w_mar.update_layout(
    title='MAR FLAT-WHEAT-AR-FOB UPR-11.5-USD-MT VS MATIF',
    xaxis_title='',
    yaxis_title='Price (USD)',
    template='plotly_white',
    height=700,
    width=1800,
    yaxis=dict(range=[-40, None]),
    legend_title='Contract Year',
    hovermode='x unified',
    xaxis=dict(
        range=[pd.Timestamp("2000-12-01"), pd.Timestamp("2001-02-28")],hoverformat='%d-%b'
    )
    
)

start_axis = pivot_plotly.index.min()
end_axis = pivot_plotly.index.max()

# Create all dates between start and end
all_dates = pd.date_range(start=start_axis, end=end_axis, freq='D')

# Select only the 1st and 15th of each month
tickvals = [d for d in all_dates if d.day in (1, 15)]
ticktext = [d.strftime('%d-%b') for d in tickvals]

# Now apply to your plot
w_mar.update_xaxes(tickvals=tickvals, ticktext=ticktext, tickangle=45)

# Show w_marure
w_mar.show()

w_mar_html = w_mar.to_html(include_plotlyjs='cdn', full_html=True)

In [0]:
# dec_contracts = merged_df[merged_df['contract_month'] == 12].copy()

# pivot = dec_contracts.pivot_table(
#     index='virtual_date',
#     columns='contract_year',
#     values='Basis',
#     aggfunc='mean'  # or 'last', 'first', 'max', etc., depending on what makes sense
# )

# # Step 1: Create a complete weekday date range for virtual year
# full_virtual_range = pd.date_range(start="2000-1-01", end="2000-12-31", freq="B")  # 'B' = business day

# # Step 2: Reindex the pivot table to this full range
# pivot = pivot.reindex(full_virtual_range)
# # Ensure all columns are numeric
# pivot = pivot.apply(pd.to_numeric, errors='coerce')
# # Optional Step 3: Interpolate to fill the gaps (linear or other methods)
# pivot_interp = pivot.interpolate(method='linear', limit_direction='both')
# # pivot_interp = pivot_interp.drop(columns=2020, errors='ignore')



# # # Drop 2020 from the interpolated pivot
# # pivot_plotly = pivot_interp.drop(columns=2020, errors='ignore')
# # cutoff_date = pd.Timestamp("2000-08-04")

# # # Mask pre-cutoff values for contract year 2021
# # if 2021 in pivot_plotly.columns:
# #     pivot_plotly.loc[pivot_plotly.index < cutoff_date, 2021] = np.nan

# # Compute average, min, max
# mean_curve = pivot_plotly.mean(axis=1)


# # Create figure
# w_dec = go.Figure()

# # Add each contract year as a trace
# for year in pivot_plotly.columns:
#     w_dec.add_trace(go.Scatter(
#         x=pivot_plotly.index,
#         y=pivot_plotly[year],
#         mode='lines',
#         name=str(year),
#         line=dict(width=1),
#         opacity=0.8,
#         hovertemplate='%{x|%d-%b}<br>%{y:.2f}<extra>%{fullData.name}</extra>'
#     ))

# # Add average curve
# w_dec.add_trace(go.Scatter(
#     x=pivot_plotly.index,
#     y=mean_curve,
#     mode='lines',
#     name='Average',
#     line=dict(color='black', width=2.5),
#     hovertemplate='%{x|%d-%b}<br>%{y:.2f}<extra>%{fullData.name}</extra>'
# ))



# # Update layout
# w_dec.update_layout(
#     title='DEC FLAT-WHEAT-AR-FOB UPR-11.5-USD-MT VS MATIF',
#     xaxis_title='',
#     yaxis_title='Price (USD)',
#     template='plotly_white',
#     height=700,
#     width=1800,
#     legend_title='Contract Year',
#     hovermode='x unified',
#     xaxis=dict(hoverformat='%d-%b')
# )


# start_axis = pivot_plotly.index.min()
# end_axis = pivot_plotly.index.max()

# # Create all dates between start and end
# all_dates = pd.date_range(start=start_axis, end=end_axis, freq='D')

# # Select only the 1st and 15th of each month
# tickvals = [d for d in all_dates if d.day in (1, 15)]
# ticktext = [d.strftime('%d-%b') for d in tickvals]

# # Now apply to your plot
# w_dec.update_xaxes(tickvals=tickvals, ticktext=ticktext, tickangle=45)

# # Show w_decure
# w_dec.show()


# w_dec_html = w_dec.to_html(include_plotlyjs='cdn', full_html=True)


# # sp_mgr.save_pd_to_excel('/sites/grainsargprojects/models/zema_prices/WHEAT/wheat_UPR_dec_vs_matif_dec.xlsx',pivot_plotly,index=False)


In [0]:
arg_fob_wheat_upbb_DEC=merged_df[merged_df['contract_month']==12]

arg_fob_wheat_upbb_DEC['Season'] = 'DEC'+ arg_fob_wheat_upbb_DEC['contract_year'].astype(str)

arg_fob_wheat_upbb_DEC = arg_fob_wheat_upbb_DEC[arg_fob_wheat_upbb_DEC['value'] != 0]

arg_fob_wheat_upbb_DEC = arg_fob_wheat_upbb_DEC.sort_values(by='date')




In [0]:
arg_fob_wheat_upbb_DEC=merged_df[merged_df['contract_month']==12]

arg_fob_wheat_upbb_DEC['Season'] = 'DEC'+ arg_fob_wheat_upbb_DEC['contract_year'].astype(str)

arg_fob_wheat_upbb_DEC = arg_fob_wheat_upbb_DEC[arg_fob_wheat_upbb_DEC['value'] != 0]

arg_fob_wheat_upbb_DEC = arg_fob_wheat_upbb_DEC.sort_values(by='date')

arg_fob_wheat_upbb_DEC=arg_fob_wheat_upbb_DEC[arg_fob_wheat_upbb_DEC['contract_year'] != 2019]



# Create empty figure
DEC_arg_wheat_upbb = go.Figure()

# Assign consistent colors for each season
colors = px.colors.qualitative.Plotly  
season_list = sorted(arg_fob_wheat_upbb_DEC['Season'].unique())
season_color_map = {season: colors[i % len(colors)] for i, season in enumerate(season_list)}

# Add FOB traces (solid lines)
for season in season_list:
    season_data = arg_fob_wheat_upbb_DEC[arg_fob_wheat_upbb_DEC['Season'] == season]
    DEC_arg_wheat_upbb.add_trace(go.Scatter(
        x=season_data['virtual_date'],
        y=season_data['value'],
        mode='lines',
        name=f"FOB {season}",
        line=dict(color=season_color_map[season], width=2, dash='solid'),
        yaxis="y1"
    ))

# Add Premium traces (dashed lines)
for season in season_list:
    season_data = arg_fob_wheat_upbb_DEC[arg_fob_wheat_upbb_DEC['Season'] == season]
    DEC_arg_wheat_upbb.add_trace(go.Scatter(
        x=season_data['virtual_date'],
        y=season_data['Basis'],
        mode='lines',
        name=f"Premium {season}",
        line=dict(color=season_color_map[season], width=2, dash='dash'),
        yaxis="y2"
    ))

# Layout with dual y-axes
DEC_arg_wheat_upbb.update_layout(
    title='DEC UPR FOB FLAT + PREMIUM',
    xaxis=dict(
        tickformat='%b',
        dtick="M1",
        hoverformat='%d-%b'
    ),
    yaxis=dict(
        title='FOB Spot Price',
    ),
    yaxis2=dict(
        title='Premium',
        overlaying='y',
        side='right',
        showgrid=False
    ),
    template='plotly_white',
    height=700,
    width=1800,
    hovermode='x unified'
)

# --- Add Buttons for filtering by Season ---
buttons = []
for season in season_list:
    visible = []
    for trace in DEC_arg_wheat_upbb.data:
        if season in trace.name:  # show FOB + Premium for that season
            visible.append(True)
        else:
            visible.append(False)
    buttons.append(dict(
        label=season,
        method="update",
        args=[{"visible": visible}]
    ))

# Add ALL button to reset
buttons.insert(0, dict(
    label="ALL",
    method="update",
    args=[{"visible": [True] * len(DEC_arg_wheat_upbb.data)}]
))

DEC_arg_wheat_upbb.update_layout(
    updatemenus=[dict(
        type="buttons",
        direction="left",
        x=0,
        y=1.15,
        xanchor="left",
        yanchor="top",
        buttons=buttons,
        showactive=True
    )]
)

DEC_arg_wheat_upbb.show()

# Export to HTML if needed
DEC_arg_wheat_upbb_html = DEC_arg_wheat_upbb.to_html(include_plotlyjs='cdn', full_html=True)



## SPOT

In [0]:

# Ensure datetime parsing
arg['date'] = pd.to_datetime(arg['date'])
arg['virtual_date'] = pd.to_datetime(arg['virtual_date'])

# Create contract_date to identify the earliest available contract
arg['contract_date'] = pd.to_datetime(dict(year=arg['contract_year'], month=arg['contract_month'], day=1))

# For each real date, get the earliest contract
spot_df = arg.sort_values(['date', 'contract_date']).groupby('date').first().reset_index()


In [0]:
def assign_season(row):
    date = row['date']
    
    # Check if the date is from Nov to Oct (for the season)
    if date.month >= 11:  # Nov and Dec belong to the current season
        season_start = date.year
    else:  # Jan to Oct belong to the previous season
        season_start = date.year - 1
    
    # Define the season in 'YYYY/YYYY' format
    season = f"{season_start}/{season_start + 1}"
    return season

# Apply the function to create the 'Season' column
spot_df['Season'] = spot_df.apply(assign_season, axis=1)

spot_df


In [0]:


# Assuming df is your DataFrame and 'virtual_date' is in datetime format
spot_df['virtual_date'] = pd.to_datetime(spot_df['virtual_date'])

# Function to adjust the virtual_date based on the condition
def adjust_virtual_date(row):
    if row['virtual_date'].month in [11, 12]:
        # Subtract 1 year from the year if month is November or December
        return row['virtual_date'].replace(year=row['virtual_date'].year - 1)
    else:
        # Keep the date as is if the month is not November or December
        return row['virtual_date']

# Apply the function to adjust the 'virtual_date'
spot_df['virtual_date'] = spot_df.apply(adjust_virtual_date, axis=1)

# Sort data by season and virtual_date
spot_df = spot_df.sort_values(['Season', 'virtual_date'])

In [0]:
import plotly.express as px

# Filter out the rows where 'Season' is '2020/2021'
spot_df_filtered = spot_df[spot_df['Season'] != '2019/2020']

# Plot by Season, excluding 2020 season
spot = px.line(
    spot_df_filtered, 
    x='virtual_date', 
    y='value', 
    color='Season', 
    title='Wheat 11.5 FOB UPR SPOT', 
    labels={'virtual_date': '', 'value': 'Spot Price', 'Season': 'Season'},
)

spot.update_traces(hovertemplate='%{x|%d-%b}<br>%{y:.2f}<extra>%{fullData.name}</extra>')
# Customize the layout for a nicer plot
spot.update_layout(
    title='Wheat 11.5 FOB UPR SPOT',
    xaxis=dict(
        title='Date', 
        showgrid=True,  # Show grid on x-axis
        gridcolor='lightgray',  # Light grid color for clarity
        tickformat='%b %d',  # Display month and day on the x-axis
        dtick="M1",  # Show ticks every month,
        hoverformat='%d-%b'
    ),
    yaxis=dict(
        title='Spot Price',
        showgrid=True,
        gridcolor='lightgray',  # Light grid color for clarity
    ),
    template='plotly_white',  # Use the white theme for a clean look
    legend_title='Season',  # Title for the legend
    height=700,
    width=1800,hovermode='x unified'
)

# Show the plot
spot.show()

# sp_mgr.save_pd_to_excel('/sites/grainsargprojects/models/zema_prices/WHEAT/wheat_spot_FOB_UPR.xlsx',spot_df_filtered,index=False)

spot_html = spot.to_html(include_plotlyjs='cdn', full_html=True)



## SPOT VS MATIF

In [0]:
spot_df

spot_df = spot_df.rename(columns={
    'value': 'spot'})

    
def assign_contract(row):
    month = row['date'].month
    year = row['date'].year
    if month in [1, 2]:
        return year, 3
    elif month in [3, 4]:
        return year, 5
    elif month in [5, 6, 7, 8]:
        return year, 9
    elif month in [9, 10, 11]:
        return year, 12
    elif month == 12:
        return year + 1, 3

# Apply function to spot df
spot_df[['target_year', 'target_month']] = spot_df.apply(assign_contract, axis=1, result_type="expand")

# Now, merge spot and matif_raw on date, contract_year (target_year), and contract_month (target_month)
spot_vs_matif = pd.merge(
    spot_df,
    matif_raw,
    how='left',
    left_on=['date', 'target_year', 'target_month'],
    right_on=['date', 'contract_year', 'contract_month'],
    suffixes=('_spot', '_matif')
)
spot_vs_matif['spot_vs_matif']=spot_vs_matif['spot']-spot_vs_matif['matif_usd']
spot_vs_matif

In [0]:
spot_df_basis=spot_df[['date','Season','spot','virtual_date']]

In [0]:
spot_vs_matif = spot_vs_matif[spot_vs_matif['Season'] != '2019/2020']
spot_matif = px.line(
    spot_vs_matif,
    x='virtual_date',
    y='spot_vs_matif',
    color='Season',  # one curve per season
    title='Matif vs Spot',
    labels={
        'virtual_date': '',
        'spot_vs_matif': 'SPOT UPR WHEAT FOB - MATIF',
        'Season': 'Season'
    },
    
)

# 4. Update layout to:
#    - Connect gaps
#    - Show months only on the x-axis
spot_matif.update_traces(connectgaps=True,hovertemplate='%{x|%d-%b}<br>%{y:.2f}<extra>%{fullData.name}</extra>')

spot_matif.update_layout(
    xaxis_title="Month",
    yaxis_title="Spot Upr Wheat - Matif (USD)",
    legend_title="Season",
    template="plotly_white",
    xaxis=dict(
        tickformat="%b",  # Only show month abbreviations like Jan, Feb, etc.
        dtick="M1",
        hoverformat='%d-%b'       # Show one tick per month
    ),
    height=700,
    width=1800,
    hovermode='x unified'
)

spot_matif.show()

spot_matif_html = spot_matif.to_html(include_plotlyjs='cdn', full_html=True)


In [0]:
# sp_mgr.save_pd_to_excel('/sites/grainsargprojects/models/zema_prices/WHEAT/wheat_spot_UPR_VS_matif.xlsx',spot_vs_matif,index=False)

In [0]:
spot_vs_matif['spot_usdc_bu']=spot_vs_matif['spot']*(100/36.7437)

#### SPOT VS HRW

In [0]:
hrw=zema.get_curve(curve="P-FUTURE-CBOT-INPUT-HRW Wheat-USDc-BU", period=f"{start_date}::{end_date}")
hrw=hrw[hrw['observation']=='Settle']
hrw=hrw[['date','value','contract_year','contract_month']]

hrw.rename(columns={'value': 'hrw_usd_bu'}, inplace=True)
hrw


In [0]:
spot_df = spot_df.rename(columns={
    'value': 'spot'})


In [0]:
def assign_contract(row):
    date = row['date']
    year = date.year
    month = date.month
    day = date.day

    if (month == 11 and day >= 11) or (month == 12) or (month == 2 and day <= 10) or (month == 1):
        return (year + 1, 3) if (month in [11, 12]) else (year, 3)
    elif (month == 2 and day >= 11) or (month == 3) or (month == 4 and day <= 10):
        return (year, 5)
    elif (month == 4 and day >= 11) or (month == 5) or (month == 6 and day <= 10):
        return (year, 7)
    elif (month == 6 and day >= 11) or (month == 7) or (month == 8 and day <= 10):
        return (year, 9)
    elif (month == 8 and day >= 11) or (month == 9) or (month == 10) or (month == 11 and day <= 10):
        return (year, 12)
    else:
        # Fallback (optional): you can raise an error or assign a default
        return (None, None)

spot_df[['target_year', 'target_month']] = spot_df.apply(assign_contract, axis=1, result_type="expand")

# Now, merge spot and matif_raw on date, contract_year (target_year), and contract_month (target_month)
spot_vs_hrw = pd.merge(
    spot_df,
    hrw,
    how='left',
    left_on=['date', 'target_year', 'target_month'],
    right_on=['date', 'contract_year', 'contract_month'],
    suffixes=('_spot', '_matif')
)

spot_vs_hrw['spot_usdc_bu']=spot_vs_hrw['spot']*(100/36.7437)

In [0]:
spot_vs_hrw['ARG_vs_HRW']=spot_vs_hrw['spot_usdc_bu']-spot_vs_hrw['hrw_usd_bu']


In [0]:
spot_vs_hrw = spot_vs_hrw[spot_vs_hrw['Season'] != '2019/2020']
# spot_vs_hrw = spot_vs_hrw[spot_vs_hrw['Season'] != '2021/2022']
spot_hrw = px.line(
    spot_vs_hrw,
    x='virtual_date',
    y='ARG_vs_HRW',
    color='Season',  # one curve per season
    title='ARG vs HRW',
    labels={
        'virtual_date': '',
        'spot_vs_hrw': 'ARG UPR Spot - HRW (USD)',
        'Season': 'Season'
    }
)

# 4. Update layout to:
#    - Connect gaps
#    - Show months only on the x-axis
spot_hrw.update_traces(connectgaps=True,hovertemplate='%{x|%d-%b}<br>%{y:.2f}<extra>%{fullData.name}</extra>')

spot_hrw.update_layout(
    xaxis_title="",
    yaxis_title="Spot ARG UPR - HRW",
    legend_title="Season",
    template="plotly_white",
    title="",
    xaxis=dict(
        tickformat="%b",  # Only show month abbreviations like Jan, Feb, etc.
        dtick="M1",
        hoverformat='%d-%b'        # Show one tick per month
    ),
    height=700,
    width=1800,
    hovermode='x unified'
    
)


spot_hrw_html = spot_hrw.to_html(include_plotlyjs='cdn', full_html=True)

spot_hrw.show()

In [0]:
# sp_mgr.save_pd_to_excel('/sites/grainsargprojects/models/zema_prices/WHEAT/wheat_spot_UPR_vs_hrw.xlsx',spot_vs_hrw,index=False)

In [0]:
spot_vs_hrw_ref=spot_vs_hrw[['date','Season','ARG_vs_HRW','virtual_date']]
spot_vs_matif_ref=spot_vs_matif[['date','Season','spot_vs_matif','virtual_date']]

In [0]:

spot_vs_ref = pd.merge(spot_vs_hrw_ref, spot_vs_matif_ref, on=['date', 'Season','virtual_date'], how='outer')

# Convert 'date' to datetime if it's not already
spot_vs_ref['date'] = pd.to_datetime(spot_vs_ref['date'])

# Create a new column 'spot_vs_ref' based on the date
spot_vs_ref['spot_vs_ref'] = spot_vs_ref.apply(
    lambda row: row['spot_vs_matif'] if (row['date'].month == 11 or 
                                          row['date'].month == 12 or 
                                          (row['date'].month == 2 and row['date'].day <= 15)) 
    else row['ARG_vs_HRW'], axis=1
)


In [0]:
spot_ref = px.line(
    spot_vs_ref,
    x='virtual_date',
    y='spot_vs_ref',
    color='Season',  # one curve per season
    title='ARG vs Ref',
    labels={
        'virtual_date': '',
        'spot_vs_hrw': 'Spot - ref market',
        'Season': 'Season'
    },

)

# 4. Update layout to:
#    - Connect gaps
#    - Show months only on the x-axis
spot_ref.update_traces(connectgaps=True,hovertemplate='%{x|%d-%b}<br>%{y:.2f}<extra>%{fullData.name}</extra>')

spot_ref.update_layout(
    xaxis_title="",
    yaxis_title="Spot ARG - Ref Market",
    legend_title="Season",
    template="plotly_white",
    title="",
    xaxis=dict(
        tickformat="%b",  # Only show month abbreviations like Jan, Feb, etc.
        dtick="M1",
        hoverformat='%d-%b'        # Show one tick per month
    ),
    height=700,
    width=1800,
    hovermode='x unified'
)

spot_ref_html = spot_ref.to_html(include_plotlyjs='cdn', full_html=True)

spot_ref.show()

In [0]:
# sp_mgr.save_pd_to_excel('/sites/grainsargprojects/models/zema_prices/WHEAT/wheat_spot_UPR_vs_ref.xlsx',spot_vs_ref,index=False)

# CORN

## MATIF

In [0]:
matif_corn= zema.get_curve(curve="P-FUTURE-ENXT-INPUT-CORN-EUR-MT", period=f"{start_date}::{end_date}")

matif_corn=matif_corn[matif_corn['observation']=='Settle']
matif_corn.rename(columns={'value': 'matif_eur'}, inplace=True)
matif_corn=matif_corn[['date','matif_eur','contract_year','contract_month']]

merged_corn = pd.merge(
    matif_corn,
    eurusd,
    on=['date', 'contract_year', 'contract_month'],
    how='inner'  # You can also use 'outer', 'left', or 'right' depending on your needs
)
merged_corn['matif_usd']=merged_corn['matif_eur']*merged_corn['eurusd']
merged_corn=merged_corn[['date','matif_usd','contract_year','contract_month']]



In [0]:
matif_raw_corn=merged_corn.copy()

## ARG UPR

In [0]:
arg_corn= zema.get_curve(curve="P-CASH-LDC-CALC-FLAT-CORN-AR-FOB Up River Arg-USD-MT", period=f"{start_date}::{end_date}")
arg_corn=arg_corn[arg_corn['observation']=='Last']
arg_corn=arg_corn[['date','value','contract_year','contract_month']]
arg_corn['day']=arg_corn['date'].dt.day
arg_corn['month']=arg_corn['date'].dt.month
arg_corn['year']=arg_corn['date'].dt.year
arg_corn['virtual_date'] = pd.to_datetime(
    {'year': 2000, 'month': arg_corn['month'], 'day': arg_corn['day']},
    errors='coerce'  # This will convert invalid dates (e.g. Feb 30) to NaT
)
# Drop rows with invalid virtual dates
arg_corn = arg_corn.dropna(subset=['virtual_date'])

# You can now sort or use this date for seasonal charts
arg_corn = arg_corn.sort_values(by='virtual_date')
arg_corn=arg_corn[['date','value','contract_year','contract_month','virtual_date']]



In [0]:
merged_corn_matif = pd.merge(
    arg_corn,
    merged_corn,
    on=['date', 'contract_year', 'contract_month'],
    how='inner'  # You can also use 'outer', 'left', or 'right' depending on your needs
)

merged_corn_matif['Basis']=merged_corn_matif['value']-merged_corn_matif['matif_usd']

In [0]:
def assign_season_corn(row):
    date = row['date']
    season_start = date.year
    return f"{season_start}/{season_start + 1}"
  
def adjust_virtual_date_corn(row):
    if row['virtual_date'].month in [1, 2]:
        # Subtract 1 year from the year if month is November or December
        return row['virtual_date']
    else:
        # Keep the date as is if the month is not November or December
        return row['virtual_date'].replace(year=row['virtual_date'].year - 1)
    

      

In [0]:
prem_corn= zema.get_curve(curve="P-CASH-LDC-INPUT-PREMIUM-CORN-AR-FOB Up River Arg-USDc-Bu", period=f"{start_date}::{end_date}")
prem_corn=prem_corn[prem_corn['observation']=='Last']
prem_corn=prem_corn[['date','value','contract_year','contract_month']]
prem_corn['day']=prem_corn['date'].dt.day
prem_corn['month']=prem_corn['date'].dt.month
prem_corn['year']=prem_corn['date'].dt.year
prem_corn['virtual_date'] = pd.to_datetime(
    {'year': 2000, 'month': prem_corn['month'], 'day': prem_corn['day']},
    errors='coerce'  # This will convert invalid dates (e.g. Feb 30) to NaT
)
# Drop rows with invalid virtual dates
prem_corn = prem_corn.dropna(subset=['virtual_date'])

# You can now sort or use this date for seasonal charts
prem_corn = prem_corn.sort_values(by='virtual_date')
prem_corn=prem_corn[['date','value','contract_year','contract_month','virtual_date']]

prem_corn['contract_date'] = pd.to_datetime(dict(year=prem_corn['contract_year'], month=prem_corn['contract_month'], day=1))

prem_corn['virtual_date'] = prem_corn.apply(adjust_virtual_date_corn, axis=1)

# For each real date, get the earliest contract
prem_corn_spot = prem_corn.sort_values(['date', 'contract_date']).groupby('date').first().reset_index()


# Add a season label (year of the real date)
prem_corn_spot['Season'] = prem_corn_spot.apply(assign_season_corn, axis=1)


### CBOT SPOT

In [0]:
cbot_corn= zema.get_curve(curve="P-FUTURE-CBOT-INPUT-CORN-USDc-BU", period=f"{start_date}::{end_date}")
cbot_corn=cbot_corn[cbot_corn['observation']=='Settle']
cbot_corn=cbot_corn[['date','value','contract_year','contract_month']]
cbot_corn['day']=cbot_corn['date'].dt.day
cbot_corn['month']=cbot_corn['date'].dt.month
cbot_corn['year']=cbot_corn['date'].dt.year

In [0]:
cbot_corn['date'] = pd.to_datetime(cbot_corn['date'])

# Define the contract months
contract_months = [3, 5, 7, 9, 12]

# Step 1: Expand to get a DataFrame with unique dates
unique_dates = cbot_corn['date'].unique()
spot_rows = []

for current_date in unique_dates:
  current_date = pd.Timestamp(current_date)
  current_month = current_date.month
  current_year = current_date.year
  
  # Find the next valid contract month (skip current month)
  future_months = [m for m in contract_months if m > current_month]
  if not future_months:
      # If we're in December, next contract is March of next year
      next_month = 3
      next_year = current_year + 1
  else:
      next_month = future_months[0]
      next_year = current_year

  # Filter the contract
  match = cbot_corn[
      (cbot_corn['date'] == current_date) &
      (cbot_corn['contract_month'] == next_month) &
      (cbot_corn['contract_year'] == next_year)
  ]
  
  if not match.empty:
      spot_rows.append(match.iloc[0])

# Combine results
spot_cbot_corn = pd.DataFrame(spot_rows).reset_index(drop=True)

In [0]:
spot_cbot_corn

spot_cbot_corn['virtual_date'] = pd.to_datetime(
    {'year': 2000, 'month': spot_cbot_corn['month'], 'day': spot_cbot_corn['day']},
    errors='coerce'  # This will convert invalid dates (e.g. Feb 30) to NaT
)
  
spot_cbot_corn['Season'] = spot_cbot_corn.apply(assign_season_corn, axis=1)

### CBOT DEC

In [0]:
# Ensure date column is datetime
cbot_corn['date'] = pd.to_datetime(cbot_corn['date'])

# Make sure contract_month and contract_year are int
cbot_corn['contract_month'] = cbot_corn['contract_month'].astype(int)
cbot_corn['contract_year'] = cbot_corn['contract_year'].astype(int)

# Container for result rows
december_contracts = []

# Loop through each unique trading date
for current_date in sorted(cbot_corn['date'].unique()):
    current_date = pd.Timestamp(current_date)
    current_year = current_date.year
    current_month = current_date.month


    # Filter for first available December contract
    match = cbot_corn[
        (cbot_corn['date'] == current_date) &
        (cbot_corn['contract_month'] == 12) &
        (cbot_corn['contract_year'] == current_year)
    ]
    
    if not match.empty:
        december_contracts.append(match.iloc[0])

# Final DataFrame with December contract evolution
december_cbot = pd.DataFrame(december_contracts).reset_index(drop=True)

In [0]:
december_cbot.rename(columns={'value': 'dec_cbot'}, inplace=True)
december_cbot=december_cbot[['date','dec_cbot']]


In [0]:
cbot_spot_dec=pd.merge(december_cbot,spot_cbot_corn,on=['date'])


### UPR CORN 

In [0]:
def assign_season_corn(row):
    date = row['date']
    if date.month >= 3:  # March to December → same year
        season_start = date.year
    else:  # January, February → previous year's marketing season
        season_start = date.year - 1
    return f"{season_start}/{season_start + 1}"
  
def adjust_virtual_date_corn(row):
    if row['virtual_date'].month in [1, 2]:
        # Subtract 1 year from the year if month is November or December
        return row['virtual_date']
    else:
        # Keep the date as is if the month is not November or December
        return row['virtual_date'].replace(year=row['virtual_date'].year - 1)

In [0]:
prem_corn= zema.get_curve(curve="P-CASH-LDC-INPUT-PREMIUM-CORN-AR-FOB Up River Arg-USDc-Bu", period=f"{start_date}::{end_date}")
prem_corn=prem_corn[prem_corn['observation']=='Last']
prem_corn=prem_corn[['date','value','contract_year','contract_month']]
prem_corn['day']=prem_corn['date'].dt.day
prem_corn['month']=prem_corn['date'].dt.month
prem_corn['year']=prem_corn['date'].dt.year
prem_corn['virtual_date'] = pd.to_datetime(
    {'year': 2000, 'month': prem_corn['month'], 'day': prem_corn['day']},
    errors='coerce'  # This will convert invalid dates (e.g. Feb 30) to NaT
)

# Drop rows with invalid virtual dates
prem_corn = prem_corn.dropna(subset=['virtual_date'])

# You can now sort or use this date for seasonal charts
prem_corn=prem_corn[['date','value','contract_year','contract_month','virtual_date']]

prem_corn['contract_date'] = pd.to_datetime(dict(year=prem_corn['contract_year'], month=prem_corn['contract_month'], day=1))

prem_corn['virtual_date'] = prem_corn.apply(adjust_virtual_date_corn, axis=1)

# Extract day and month
prem_corn['day'] = prem_corn['date'].dt.day
prem_corn['month'] = prem_corn['date'].dt.month
prem_corn['year'] = prem_corn['date'].dt.year

# Define target contract month/year based on the date rules
def get_spot_contract_info(row):
    if row['day'] < 15:
        target_month = row['month']
        target_year = row['year']
    else:
        if row['month'] == 12:
            target_month = 1
            target_year = row['year'] + 1
        else:
            target_month = row['month'] + 1
            target_year = row['year']
    return pd.Series({'target_month': target_month, 'target_year': target_year})

# Apply to get target contract month and year
prem_corn[['target_month', 'target_year']] = prem_corn.apply(get_spot_contract_info, axis=1)

# Filter for rows where contract_month and contract_year match target
spot_prem_corn = prem_corn[(prem_corn['contract_month'] == prem_corn['target_month']) & 
             (prem_corn['contract_year'] == prem_corn['target_year'])]

spot_prem_corn=spot_prem_corn[['date','value','virtual_date']]
spot_corn_upr_prem=spot_prem_corn.copy()

In [0]:
spot_corn_upr_prem['Season'] = spot_corn_upr_prem.apply(assign_season_corn, axis=1)

In [0]:
spot_corn_upr_prem.rename(columns={'value': 'UPR_prem'}, inplace=True)


In [0]:
spot_corn_upr_prem_merged=spot_corn_upr_prem[['date','UPR_prem']]


In [0]:
cbot_spot_dec_upr=pd.merge(cbot_spot_dec,spot_corn_upr_prem_merged,on=['date'])
cbot_spot_dec_upr.rename(columns={'value': 'spot_cbot'}, inplace=True)



In [0]:
# Make sure 'virtual_date' is datetime
cbot_spot_dec_upr['virtual_date'] = pd.to_datetime(cbot_spot_dec_upr['virtual_date'])


# Define line styles for each variable
line_styles = {
    'dec_cbot': 'solid',
    'spot_cbot': 'dot',
    'UPR_prem': 'dash'
}

# Create a unique color for each season
season_colors = px.colors.qualitative.Set1
seasons = cbot_spot_dec_upr['Season'].unique()
season_color_map = {season: season_colors[i % len(season_colors)] for i, season in enumerate(seasons)}

# Manually override for specific seasons
season_color_map['2025/2026'] = 'red'
season_color_map['2024/2025'] = 'black'

# Initialize figure
cbot_vs_upr = go.Figure()

# Plot each season separately
for season in seasons:
    cbot_spot_dec_upr_season = cbot_spot_dec_upr[cbot_spot_dec_upr['Season'] == season]
    cbot_spot_dec_upr_season = cbot_spot_dec_upr[cbot_spot_dec_upr['Season'] == season].sort_values('virtual_date')
    color = season_color_map[season]

    # Add dec_cbot
    cbot_vs_upr.add_trace(go.Scatter(
        x=cbot_spot_dec_upr_season['virtual_date'],
        y=cbot_spot_dec_upr_season['dec_cbot'],
        mode='lines',
        name=f'Dec CBOT {season}',
        line=dict(color=color, dash=line_styles['dec_cbot']),
        yaxis='y1',
        hovertemplate=(
        'Date: %{x|%d %b}<br>' +
        'Season: ' + season + '<br>' +
        'Value: %{y}<extra></extra>'
    )
    ))

    # Add spot_cbot
    cbot_vs_upr.add_trace(go.Scatter(
        x=cbot_spot_dec_upr_season['virtual_date'],
        y=cbot_spot_dec_upr_season['spot_cbot'],
        mode='lines',
        name=f'Spot CBOT {season}',
        line=dict(color=color, dash=line_styles['spot_cbot']),
        yaxis='y1',
        hovertemplate=(
        'Date: %{x|%d %b}<br>' +
        'Season: ' + season + '<br>' +
        'Value: %{y}<extra></extra>'
    )
    ))

    # Add UPR premium on secondary axis
    cbot_vs_upr.add_trace(go.Scatter(
        x=cbot_spot_dec_upr_season['virtual_date'],
        y=cbot_spot_dec_upr_season['UPR_prem'],
        mode='lines',
        name=f'UPR Premium {season}',
        line=dict(color=color, dash=line_styles['UPR_prem']),
        yaxis='y2',
        hovertemplate=(
        'Date: %{x|%d %b}<br>' +
        'Season: ' + season + '<br>' +
        'Value: %{y}<extra></extra>'
        )
    ))

# Update layout
cbot_vs_upr.update_layout(
    title='CBOT Prices and UPR Premium by Season',
    xaxis=dict(
        tickformat='%b',  # Show abbreviated month names like Jan, Feb, etc.
        title='Date',
        showgrid=False
    ),
    yaxis=dict(
        title='CBOT Prices (USDc/bu)',
        showgrid=False
    ),
    yaxis2=dict(
        title='UPR Premium',
        overlaying='y',
        side='right',
        showgrid=False
    ),
    legend=dict(title='Legend', x=1.02, y=1),
    template='plotly_white',
    height=900,
    width=1700,
    hovermode="x unified",
)

cbot_vs_upr.show()

cbot_vs_upr_html = cbot_vs_upr.to_html(include_plotlyjs='cdn', full_html=True)

In [0]:
# sp_mgr.save_pd_to_excel('/sites/grainsargprojects/models/zema_prices/CORN/corn_spot_cbot.xlsx',cbot_corn_spot,index=False)

In [0]:
cbot_corn= zema.get_curve(curve="P-FUTURE-CBOT-INPUT-CORN-USDc-BU", period=f"{start_date}::{end_date}")
cbot_corn=cbot_corn[cbot_corn['observation']=='Settle']
cbot_corn=cbot_corn[['date','value','contract_year','contract_month']]
cbot_corn['day']=cbot_corn['date'].dt.day
cbot_corn['month']=cbot_corn['date'].dt.month
cbot_corn['year']=cbot_corn['date'].dt.year
cbot_corn['virtual_date'] = pd.to_datetime(
    {'year': 2000, 'month': cbot_corn['month'], 'day': cbot_corn['day']},
    errors='coerce'  # This will convert invalid dates (e.g. Feb 30) to NaT
)
# Drop rows with invalid virtual dates
cbot_corn = cbot_corn.dropna(subset=['virtual_date'])

# You can now sort or use this date for seasonal charts
cbot_corn = cbot_corn.sort_values(by='virtual_date')
cbot_corn=cbot_corn[['date','value','contract_year','contract_month','virtual_date']]

cbot_corn['contract_date'] = pd.to_datetime(dict(year=cbot_corn['contract_year'], month=cbot_corn['contract_month'], day=1))

cbot_corn['virtual_date'] = cbot_corn.apply(adjust_virtual_date_corn, axis=1)

In [0]:
jul_contracts_cbot_corn = cbot_corn[cbot_corn['contract_month'] == 7].copy()

jul_contracts_cbot_corn['date_year'] = jul_contracts_cbot_corn['date'].dt.year
jul_contracts_cbot_corn['year_diff'] = abs(jul_contracts_cbot_corn['contract_year'] - jul_contracts_cbot_corn['date_year'])

# Step 3: Keep only the row with the smallest year_diff for each date
jul_contracts_cbot_corn = jul_contracts_cbot_corn.sort_values('year_diff').drop_duplicates(subset='date', keep='first')

# Optional cleanup
jul_contracts_cbot_corn = jul_contracts_cbot_corn.drop(columns=['date_year', 'year_diff'])
jul_contracts_cbot_corn['Season'] = jul_contracts_cbot_corn.apply(assign_season_corn, axis=1)

jul_contracts_cbot_corn = jul_contracts_cbot_corn.sort_values(['date', 'contract_date'])

jul_contracts_cbot_corn = jul_contracts_cbot_corn[jul_contracts_cbot_corn['Season'] != '2019/2020']

jul_prem_cbot_corn = px.line(
    jul_contracts_cbot_corn,
    x='virtual_date',
    y='value',
    color='Season',
    labels={
        'virtual_date': '',
        'value': 'Spot Price',
        'season': 'Year'
    },
    title='July Contract cbot_corn',
)
jul_prem_cbot_corn.update_traces(hovertemplate='%{x|%d-%b}<br>%{y:.2f}<extra>%{fullData.name}</extra>')

jul_prem_cbot_corn.update_layout(
    xaxis=dict(
        tickformat='%b',  # Month format like Jan, Feb...
        dtick="M1",
        hoverformat='%d-%b'
    ),
    yaxis_title='Price',
    template='plotly_white',
    height=700,
    width=1800,
    hovermode='x unified'
)

jul_prem_cbot_corn.show()


jul_prem_cbot_corn_html = jul_prem_cbot_corn.to_html(include_plotlyjs='cdn', full_html=True)

# sp_mgr.save_pd_to_excel('/sites/grainsargprojects/models/zema_prices/CORN/cbot_corn_prem_jul.xlsx',jul_contracts_cbot_corn,index=False)



In [0]:
# Remove unwanted season
spot_corn_upr_prem = spot_corn_upr_prem[spot_corn_upr_prem['Season'] != '2019/2020']

# Sort by date
spot_corn_upr_prem = spot_corn_upr_prem.sort_values(by='virtual_date')

# Plot with Plotly
prem_spot_corn = px.line(
    spot_corn_upr_prem,
    x='virtual_date',
    y='UPR_prem',
    color='Season',
    labels={
        'virtual_date': '',
        'value': 'Spot Price',
        'season': 'Year'
    },
    title='Corn Spot Premium',
)

prem_spot_corn.update_traces(
    hovertemplate='%{x|%d-%b}<br>%{y:.2f}<extra>%{fullData.name}</extra>'
)
prem_spot_corn.update_layout(
    xaxis=dict(
        tickformat='%b',
        dtick="M1",
        hoverformat='%d-%b'
    ),
    yaxis_title='Price',
    template='plotly_white',
    height=700,
    width=1800,
    hovermode='x unified'
)

prem_spot_corn_html = prem_spot_corn.to_html(include_plotlyjs='cdn', full_html=True)
prem_spot_corn.show()


### BB VS UPR

In [0]:
prem_corn_bb= zema.get_curve(curve="P-CASH-LDC-INPUT-PREMIUM-CORN-AR-FOB Bahia Blanca Arg-USDc-Bu", period=f"{start_date}::{end_date}")
prem_corn_bb=prem_corn_bb[prem_corn_bb['observation']=='Last']
prem_corn_bb=prem_corn_bb[['date','value','contract_year','contract_month']]
prem_corn_bb['day']=prem_corn_bb['date'].dt.day
prem_corn_bb['month']=prem_corn_bb['date'].dt.month
prem_corn_bb['year']=prem_corn_bb['date'].dt.year
prem_corn_bb['virtual_date'] = pd.to_datetime(
    {'year': 2000, 'month': prem_corn_bb['month'], 'day': prem_corn_bb['day']},
    errors='coerce'  # This will convert invalid dates (e.g. Feb 30) to NaT
)

# Drop rows with invalid virtual dates
prem_corn_bb = prem_corn_bb.dropna(subset=['virtual_date'])

# You can now sort or use this date for seasonal charts
prem_corn_bb=prem_corn_bb[['date','value','contract_year','contract_month','virtual_date']]

prem_corn_bb['contract_date'] = pd.to_datetime(dict(year=prem_corn_bb['contract_year'], month=prem_corn_bb['contract_month'], day=1))

prem_corn_bb['virtual_date'] = prem_corn_bb.apply(adjust_virtual_date_corn, axis=1)

# Extract day and month
prem_corn_bb['day'] = prem_corn_bb['date'].dt.day
prem_corn_bb['month'] = prem_corn_bb['date'].dt.month
prem_corn_bb['year'] = prem_corn_bb['date'].dt.year

# Apply to get target contract month and year
prem_corn_bb[['target_month', 'target_year']] = prem_corn_bb.apply(get_spot_contract_info, axis=1)

# Filter for rows where contract_month and contract_year match target
spot_prem_corn_bb = prem_corn_bb[(prem_corn_bb['contract_month'] == prem_corn_bb['target_month']) & 
             (prem_corn_bb['contract_year'] == prem_corn_bb['target_year'])]

spot_prem_corn_bb=spot_prem_corn_bb.rename(columns={'value': 'prem_bb'})
spot_prem_corn_bb=spot_prem_corn_bb[['date','prem_bb']]
spot_prem_corn_bb

In [0]:
corn_spot_upr_bb=pd.merge(spot_prem_corn_bb,spot_corn_upr_prem,on='date')
corn_spot_upr_bb

In [0]:
corn_spot_upr_bb['BB VS UPR']=corn_spot_upr_bb['prem_bb']-corn_spot_upr_bb['UPR_prem']

In [0]:

# Plot with Plotly
corn_bb_upr_prem = px.line(
    corn_spot_upr_bb,
    x='virtual_date',
    y='BB VS UPR',
    color='Season',
    labels={
        'virtual_date': '',
        'BB VS UPR': 'BB - UPR',
        'season': 'Year'
    },
    title='Corn Spot BB - Corn UPR (cts/bu)',
)

corn_bb_upr_prem.update_traces(hovertemplate='%{x|%d-%b}<br>%{y:.2f}<extra>%{fullData.name}</extra>')

corn_bb_upr_prem.update_layout(
    xaxis=dict(
        tickformat='%b',  # Month format like Jan, Feb...
        dtick="M1",
        hoverformat='%d-%b'
    ),
    yaxis_title='BB - UPR',
    template='plotly_white',
    height=700,
    width=1800,
    hovermode='x unified'
)


corn_bb_upr_prem_html = corn_bb_upr_prem.to_html(include_plotlyjs='cdn', full_html=True)

corn_bb_upr_prem.show()




## CHARTS

### APRIL

In [0]:
apr_contracts_corn = prem_corn[prem_corn['contract_month'] == 4].copy()

apr_contracts_corn['date_year'] = apr_contracts_corn['date'].dt.year
apr_contracts_corn['year_diff'] = abs(apr_contracts_corn['contract_year'] - apr_contracts_corn['date_year'])

# Step 3: Keep only the row with the smallest year_diff for each date
apr_contracts_corn = apr_contracts_corn.sort_values('year_diff').drop_duplicates(subset='date', keep='first')

# Optional cleanup
apr_contracts_corn = apr_contracts_corn.drop(columns=['date_year', 'year_diff'])
apr_contracts_corn['Season'] = apr_contracts_corn.apply(assign_season_corn, axis=1)

apr_contracts_corn = apr_contracts_corn.sort_values(['date', 'contract_date'])

apr_contracts_corn = apr_contracts_corn[apr_contracts_corn['Season'] != '2019/2020']


apr_prem_corn = px.line(
    apr_contracts_corn,
    x='virtual_date',
    y='value',
    color='Season',
    labels={
        'virtual_date': '',
        'value': 'Spot Price',
        'season': 'Year'
    },
    title='APRIL CORN UPR PREMIUM',
)

apr_prem_corn.update_traces(hovertemplate='%{x|%d-%b}<br>%{y:.2f}<extra>%{fullData.name}</extra>')  
apr_prem_corn.update_layout(
    xaxis=dict(
        tickformat='%b',  # Month format like Jan, Feb...
        dtick="M1",hoverformat='%d-%b'
    ),
    yaxis_title='Price',
    template='plotly_white',
    height=700,
    width=1800,
    hovermode='x unified'
)



apr_prem_corn_html = apr_prem_corn.to_html(include_plotlyjs='cdn', full_html=True)

# sp_mgr.save_pd_to_excel('/sites/grainsargprojects/models/zema_prices/CORN/corn_prem_apr.xlsx',apr_contracts_corn,index=False)

### MAY

In [0]:
may_contracts_corn = prem_corn[prem_corn['contract_month'] == 5].copy()

may_contracts_corn['date_year'] = may_contracts_corn['date'].dt.year
may_contracts_corn['year_diff'] = abs(may_contracts_corn['contract_year'] - may_contracts_corn['date_year'])

# Step 3: Keep only the row with the smallest year_diff for each date
may_contracts_corn = may_contracts_corn.sort_values('year_diff').drop_duplicates(subset='date', keep='first')

# Optional cleanup
may_contracts_corn = may_contracts_corn.drop(columns=['date_year', 'year_diff'])
may_contracts_corn['Season'] = may_contracts_corn.apply(assign_season_corn, axis=1)

may_contracts_corn = may_contracts_corn.sort_values(['date', 'contract_date'])

may_contracts_corn = may_contracts_corn[may_contracts_corn['Season'] != '2019/2020']


may_prem_corn = px.line(
    may_contracts_corn,
    x='virtual_date',
    y='value',
    color='Season',
    labels={
        'virtual_date': '',
        'value': 'Spot Price',
        'season': 'Year'
    },
    title='MAY CORN UPR PREMIUM',
)

may_prem_corn.update_traces(hovertemplate='%{x|%d-%b}<br>%{y:.2f}<extra>%{fullData.name}</extra>')
may_prem_corn.update_layout(
    xaxis=dict(
        tickformat='%b',  # Month format like Jan, Feb...
        dtick="M1",
        hoverformat='%d-%b'
    ),
    yaxis_title='Price',
    template='plotly_white',
    height=700,
    width=1800,
    hovermode='x unified'
)


may_prem_corn_html = may_prem_corn.to_html(include_plotlyjs='cdn', full_html=True)


In [0]:
# sp_mgr.save_pd_to_excel('/sites/grainsargprojects/models/zema_prices/CORN/corn_prem_may.xlsx',may_contracts_corn,index=False)

### JULY

In [0]:
jul_contracts_corn = prem_corn[prem_corn['contract_month'] == 7].copy()

jul_contracts_corn['date_year'] = jul_contracts_corn['date'].dt.year
jul_contracts_corn['year_diff'] = abs(jul_contracts_corn['contract_year'] - jul_contracts_corn['date_year'])

# Step 3: Keep only the row with the smallest year_diff for each date
jul_contracts_corn = jul_contracts_corn.sort_values('year_diff').drop_duplicates(subset='date', keep='first')

# Optional cleanup
jul_contracts_corn = jul_contracts_corn.drop(columns=['date_year', 'year_diff'])
jul_contracts_corn['Season'] = jul_contracts_corn.apply(assign_season_corn, axis=1)

jul_contracts_corn = jul_contracts_corn.sort_values(['date', 'contract_date'])

jul_contracts_corn = jul_contracts_corn[jul_contracts_corn['Season'] != '2019/2020']

jul_prem_corn = px.line(
    jul_contracts_corn,
    x='virtual_date',
    y='value',
    color='Season',
    labels={
        'virtual_date': '',
        'value': 'Spot Price',
        'season': 'Year'
    },
    title='July Contract Corn Premium',
)
jul_prem_corn.update_traces(hovertemplate='%{x|%d-%b}<br>%{y:.2f}<extra>%{fullData.name}</extra>')

jul_prem_corn.update_layout(
    xaxis=dict(
        tickformat='%b',  # Month format like Jan, Feb...
        dtick="M1",
        hoverformat='%d-%b'
    ),
    yaxis_title='Price',
    template='plotly_white',
    height=700,
    width=1800,
    hovermode='x unified'
)



jul_prem_corn_html = jul_prem_corn.to_html(include_plotlyjs='cdn', full_html=True)

# sp_mgr.save_pd_to_excel('/sites/grainsargprojects/models/zema_prices/CORN/corn_prem_jul.xlsx',jul_contracts_corn,index=False)

### DEC

In [0]:
dec_contracts_corn = prem_corn[prem_corn['contract_month'] == 12].copy()

dec_contracts_corn['date_year'] = dec_contracts_corn['date'].dt.year
dec_contracts_corn['year_diff'] = abs(dec_contracts_corn['contract_year'] - dec_contracts_corn['date_year'])

# Step 3: Keep only the row with the smallest year_diff for each date
dec_contracts_corn = dec_contracts_corn.sort_values('year_diff').drop_duplicates(subset='date', keep='first')

# Optional cleanup
dec_contracts_corn = dec_contracts_corn.drop(columns=['date_year', 'year_diff'])
dec_contracts_corn['Season'] = dec_contracts_corn.apply(assign_season_corn, axis=1)

dec_contracts_corn = dec_contracts_corn.sort_values(['date', 'contract_date'])

dec_contracts_corn = dec_contracts_corn[dec_contracts_corn['Season'] != '2019/2020']

dec_prem_corn = px.line(
    dec_contracts_corn,
    x='virtual_date',
    y='value',
    color='Season',
    labels={
        'virtual_date': '',
        'value': 'Spot Price',
        'season': 'Year'
    },
    title='DEC CORN UPR PREMIUM',
)

dec_prem_corn.update_traces(hovertemplate='%{x|%d-%b}<br>%{y:.2f}<extra>%{fullData.name}</extra>')
dec_prem_corn.update_layout(
    xaxis=dict(
        tickformat='%b',  # Month format like Jan, Feb...
        dtick="M1",
        hoverformat='%d-%b'
    ),
    yaxis_title='Price',
    template='plotly_white',
    height=700,
    width=1800,
    hovermode='x unified'
    
)
dec_prem_corn_html = dec_prem_corn.to_html(include_plotlyjs='cdn', full_html=True)

# sp_mgr.save_pd_to_excel('/sites/grainsargprojects/models/zema_prices/CORN/corn_prem_dec.xlsx',dec_contracts_corn,index=False)

#### APRIL AND JULY PREM AND FOB, EVOLUTION STARTING CORRESPONDING MONTH

In [0]:
arg_fob_corn_upbb=zema.get_curve(curve='P-CASH-LDC-CALC-FLAT-CORN-AR-FOB Up River Arg-USD-MT', period=f"{start_date}::{end_date}")
arg_prem_corn_upbb=zema.get_curve(curve='P-CASH-LDC-INPUT-PREMIUM-CORN-AR-FOB Up River Arg-USDc-Bu', period=f"{start_date}::{end_date}")


In [0]:
arg_fob_corn_upbb=arg_fob_corn_upbb[arg_fob_corn_upbb['observation']=='Last']
arg_fob_corn_upbb=arg_fob_corn_upbb[['date','value','contract_year','contract_month']]
arg_fob_corn_upbb['day']=arg_fob_corn_upbb['date'].dt.day
arg_fob_corn_upbb['month']=arg_fob_corn_upbb['date'].dt.month
arg_fob_corn_upbb['year']=arg_fob_corn_upbb['date'].dt.year
arg_fob_corn_upbb['virtual_date'] = pd.to_datetime(
    {'year': 2000, 'month': arg_fob_corn_upbb['month'], 'day': arg_fob_corn_upbb['day']},
    errors='coerce'  # This will convert invalid dates (e.g. Feb 30) to NaT
)

# Drop rows with invalid virtual dates
arg_fob_corn_upbb = arg_fob_corn_upbb.dropna(subset=['virtual_date'])

# You can now sort or use this date for seasonal charts
arg_fob_corn_upbb = arg_fob_corn_upbb.sort_values(by='virtual_date')
arg_fob_corn_upbb=arg_fob_corn_upbb[['date','value','contract_year','contract_month','virtual_date']]


arg_prem_corn_upbb=arg_prem_corn_upbb[arg_prem_corn_upbb['observation']=='Last']
arg_prem_corn_upbb=arg_prem_corn_upbb[['date','value','contract_year','contract_month']]
arg_prem_corn_upbb['day']=arg_prem_corn_upbb['date'].dt.day
arg_prem_corn_upbb['month']=arg_prem_corn_upbb['date'].dt.month
arg_prem_corn_upbb['year']=arg_prem_corn_upbb['date'].dt.year
arg_prem_corn_upbb['virtual_date'] = pd.to_datetime(
    {'year': 2000, 'month': arg_prem_corn_upbb['month'], 'day': arg_prem_corn_upbb['day']},
    errors='coerce'  # This will convert invalid dates (e.g. Feb 30) to NaT
)

# Drop rows with invalid virtual dates
arg_prem_corn_upbb = arg_prem_corn_upbb.dropna(subset=['virtual_date'])

# You can now sort or use this date for seasonal charts
arg_prem_corn_upbb = arg_prem_corn_upbb.sort_values(by='virtual_date')
arg_prem_corn_upbb=arg_prem_corn_upbb[['date','value','contract_year','contract_month','virtual_date']]

In [0]:
arg_fob_corn_upbb_APR=arg_fob_corn_upbb[arg_fob_corn_upbb['contract_month']==4]

arg_fob_corn_upbb_APR['Season'] = 'APR'+ arg_fob_corn_upbb_APR['contract_year'].astype(str)
  
def adjust_virtual_date_sbs_APR(row):
    if row['virtual_date'].month in [1, 2,3,4]:
        # Subtract 1 year from the year if month is November or December
        return row['virtual_date'] + pd.DateOffset(years=1)
    else:
        # Keep the date as is if the month is not November or December
        return row['virtual_date'].replace(year=row['virtual_date'].year)
      
arg_fob_corn_upbb_APR['virtual_date'] = arg_fob_corn_upbb_APR.apply(adjust_virtual_date_sbs_APR, axis=1)
arg_fob_corn_upbb_APR = arg_fob_corn_upbb_APR[arg_fob_corn_upbb_APR['value'] != 0]

arg_fob_corn_upbb_APR = arg_fob_corn_upbb_APR[
    (arg_fob_corn_upbb_APR['date'].dt.month.isin([1, 2, 3,4]) & (arg_fob_corn_upbb_APR['date'].dt.year == arg_fob_corn_upbb_APR['contract_year'])) |
    (~arg_fob_corn_upbb_APR['date'].dt.month.isin([1, 2, 3,4]) & (arg_fob_corn_upbb_APR['date'].dt.year +1 == arg_fob_corn_upbb_APR['contract_year']))
]
arg_fob_corn_upbb_APR = arg_fob_corn_upbb_APR.sort_values(by='date')



arg_prem_corn_upbb_APR=arg_prem_corn_upbb[arg_prem_corn_upbb['contract_month']==4]

arg_prem_corn_upbb_APR['Season'] = 'APR'+ arg_prem_corn_upbb_APR['contract_year'].astype(str)
      
arg_prem_corn_upbb_APR['virtual_date'] = arg_prem_corn_upbb_APR.apply(adjust_virtual_date_sbs_APR, axis=1)
arg_prem_corn_upbb_APR = arg_prem_corn_upbb_APR[arg_prem_corn_upbb_APR['value'] != 0]

arg_prem_corn_upbb_APR = arg_prem_corn_upbb_APR[
    (arg_prem_corn_upbb_APR['date'].dt.month.isin([1, 2, 3,4]) & (arg_prem_corn_upbb_APR['date'].dt.year == arg_prem_corn_upbb_APR['contract_year'])) |
    (~arg_prem_corn_upbb_APR['date'].dt.month.isin([1, 2, 3,4]) & (arg_prem_corn_upbb_APR['date'].dt.year +1 == arg_prem_corn_upbb_APR['contract_year']))
]
arg_prem_corn_upbb_APR = arg_prem_corn_upbb_APR.sort_values(by='date')




# Create empty figure
APR_arg_corn_upbb = go.Figure()

# Assign consistent colors for each season
colors = px.colors.qualitative.Plotly  
season_list = sorted(arg_fob_corn_upbb_APR['Season'].unique())
season_color_map = {season: colors[i % len(colors)] for i, season in enumerate(season_list)}

# Add FOB traces (solid lines)
for season in season_list:
    season_data = arg_fob_corn_upbb_APR[arg_fob_corn_upbb_APR['Season'] == season]
    APR_arg_corn_upbb.add_trace(go.Scatter(
        x=season_data['virtual_date'],
        y=season_data['value'],
        mode='lines',
        name=f"FOB {season}",
        line=dict(color=season_color_map[season], width=2, dash='solid'),
        yaxis="y1"
    ))

# Add Premium traces (dashed lines)
for season in season_list:
    season_data = arg_prem_corn_upbb_APR[arg_prem_corn_upbb_APR['Season'] == season]
    APR_arg_corn_upbb.add_trace(go.Scatter(
        x=season_data['virtual_date'],
        y=season_data['value'],
        mode='lines',
        name=f"Premium {season}",
        line=dict(color=season_color_map[season], width=2, dash='dash'),
        yaxis="y2"
    ))

# Layout with dual y-axes
APR_arg_corn_upbb.update_layout(
    title='APR UPR FOB FLAT + PREMIUM',
    xaxis=dict(
        tickformat='%b',
        dtick="M1",
        hoverformat='%d-%b'
    ),
    yaxis=dict(
        title='FOB Spot Price',
    ),
    yaxis2=dict(
        title='Premium',
        overlaying='y',
        side='right',
        showgrid=False
    ),
    template='plotly_white',
    height=700,
    width=1800,
    hovermode='x unified'
)

# --- Add Buttons for filtering by Season ---
buttons = []
for season in season_list:
    visible = []
    for trace in APR_arg_corn_upbb.data:
        if season in trace.name:  # show FOB + Premium for that season
            visible.append(True)
        else:
            visible.append(False)
    buttons.append(dict(
        label=season,
        method="update",
        args=[{"visible": visible}]
    ))

# Add ALL button to reset
buttons.insert(0, dict(
    label="ALL",
    method="update",
    args=[{"visible": [True] * len(APR_arg_corn_upbb.data)}]
))

APR_arg_corn_upbb.update_layout(
    updatemenus=[dict(
        type="buttons",
        direction="left",
        x=0,
        y=1.15,
        xanchor="left",
        yanchor="top",
        buttons=buttons,
        showactive=True
    )]
)

APR_arg_corn_upbb.show()

# Export to HTML if needed
APR_arg_corn_upbb_html = APR_arg_corn_upbb.to_html(include_plotlyjs='cdn', full_html=True)



In [0]:
arg_fob_corn_upbb_JUL=arg_fob_corn_upbb[arg_fob_corn_upbb['contract_month']==7]

arg_fob_corn_upbb_JUL['Season'] = 'JUL'+ arg_fob_corn_upbb_JUL['contract_year'].astype(str)
  
def adjust_virtual_date_sbs_JUL(row):
    if row['virtual_date'].month in [1, 2, 3,4,5,6,7]:
        # Subtract 1 year from the year if month is November or December
        return row['virtual_date'] + pd.DateOffset(years=1)
    else:
        # Keep the date as is if the month is not November or December
        return row['virtual_date'].replace(year=row['virtual_date'].year)
      
arg_fob_corn_upbb_JUL['virtual_date'] = arg_fob_corn_upbb_JUL.apply(adjust_virtual_date_sbs_JUL, axis=1)
arg_fob_corn_upbb_JUL = arg_fob_corn_upbb_JUL[arg_fob_corn_upbb_JUL['value'] != 0]

arg_fob_corn_upbb_JUL = arg_fob_corn_upbb_JUL[
    (arg_fob_corn_upbb_JUL['date'].dt.month.isin([1, 2, 3,4,5,6,7]) & (arg_fob_corn_upbb_JUL['date'].dt.year == arg_fob_corn_upbb_JUL['contract_year'])) |
    (~arg_fob_corn_upbb_JUL['date'].dt.month.isin([1, 2, 3,4,5,6,7]) & (arg_fob_corn_upbb_JUL['date'].dt.year +1 == arg_fob_corn_upbb_JUL['contract_year']))
]
arg_fob_corn_upbb_JUL = arg_fob_corn_upbb_JUL.sort_values(by='date')



arg_prem_corn_upbb_JUL=arg_prem_corn_upbb[arg_prem_corn_upbb['contract_month']==7]

arg_prem_corn_upbb_JUL['Season'] = 'JUL'+ arg_prem_corn_upbb_JUL['contract_year'].astype(str)
      
arg_prem_corn_upbb_JUL['virtual_date'] = arg_prem_corn_upbb_JUL.apply(adjust_virtual_date_sbs_JUL, axis=1)
arg_prem_corn_upbb_JUL = arg_prem_corn_upbb_JUL[arg_prem_corn_upbb_JUL['value'] != 0]

arg_prem_corn_upbb_JUL = arg_prem_corn_upbb_JUL[
    (arg_prem_corn_upbb_JUL['date'].dt.month.isin([1, 2, 3,4,5,6,7]) & (arg_prem_corn_upbb_JUL['date'].dt.year == arg_prem_corn_upbb_JUL['contract_year'])) |
    (~arg_prem_corn_upbb_JUL['date'].dt.month.isin([1, 2, 3,4,5,6,7]) & (arg_prem_corn_upbb_JUL['date'].dt.year +1 == arg_prem_corn_upbb_JUL['contract_year']))
]
arg_prem_corn_upbb_JUL = arg_prem_corn_upbb_JUL.sort_values(by='date')




# Create empty figure
JUL_arg_corn_upbb = go.Figure()

# Assign consistent colors for each season
colors = px.colors.qualitative.Plotly  # nice strong palette
season_list = sorted(arg_fob_corn_upbb_JUL['Season'].unique())
season_color_map = {season: colors[i % len(colors)] for i, season in enumerate(season_list)}

# Add FOB traces (solid lines)
for season in season_list:
    season_data = arg_fob_corn_upbb_JUL[arg_fob_corn_upbb_JUL['Season'] == season]
    JUL_arg_corn_upbb.add_trace(go.Scatter(
        x=season_data['virtual_date'],
        y=season_data['value'],
        mode='lines',
        name=f"FOB {season}",
        line=dict(color=season_color_map[season], width=2, dash='solid'),
        yaxis="y1"
    ))

# Add Premium traces (dashed lines)
for season in season_list:
    season_data = arg_prem_corn_upbb_JUL[arg_prem_corn_upbb_JUL['Season'] == season]
    JUL_arg_corn_upbb.add_trace(go.Scatter(
        x=season_data['virtual_date'],
        y=season_data['value'],
        mode='lines',
        name=f"Premium {season}",
        line=dict(color=season_color_map[season], width=2, dash='dash'),
        yaxis="y2"
    ))

# Layout with dual y-axes
JUL_arg_corn_upbb.update_layout(
    title='JUL UPR FOB FLAT + PREMIUM',
    xaxis=dict(
        tickformat='%b',
        dtick="M1",
        hoverformat='%d-%b'
    ),
    yaxis=dict(
        title='FOB Spot Price',
    ),
    yaxis2=dict(
        title='Premium',
        overlaying='y',
        side='right',
        showgrid=False
    ),
    template='plotly_white',
    height=700,
    width=1800,
    hovermode='x unified'
)

# --- Add Buttons for filtering by Season ---
buttons = []
for season in season_list:
    visible = []
    for trace in JUL_arg_corn_upbb.data:
        if season in trace.name:  # show FOB + Premium for that season
            visible.append(True)
        else:
            visible.append(False)
    buttons.append(dict(
        label=season,
        method="update",
        args=[{"visible": visible}]
    ))

# Add ALL button to reset
buttons.insert(0, dict(
    label="ALL",
    method="update",
    args=[{"visible": [True] * len(JUL_arg_corn_upbb.data)}]
))

JUL_arg_corn_upbb.update_layout(
    updatemenus=[dict(
        type="buttons",
        direction="left",
        x=0,
        y=1.15,
        xanchor="left",
        yanchor="top",
        buttons=buttons,
        showactive=True
    )]
)

JUL_arg_corn_upbb.show()

# Export to HTML if needed
JUL_arg_corn_upbb_html = JUL_arg_corn_upbb.to_html(include_plotlyjs='cdn', full_html=True)



In [0]:
# corn_hist_freight  = f"""
# <!DOCTYPE html>
# <html>
# <head>
#     <title>CORN APR AND JUL UPR PRICES</title>
#     <style>
#         body {{
#             font-family: Arial, sans-serif;
#             padding: 20px;
#         }}
#         h2 {{
#             margin-top: 40px;
#             color: #2c3e50;
#         }}
#         .chart-container {{
#             margin-bottom: 50px;
#         }}
#     </style>
# </head>
# <body>
#     <h1>Corn UPR</h1>
#     <h2>APRIL FOB AND PREM</h2>
#     <div class="chart-container">{APR_arg_corn_upbb_html}</div>
    
#     <h2>JUL FOB AND PREM</h2>
#     <div class="chart-container">{JUL_arg_corn_upbb_html}</div>

# </body>
# </html>
# """
# corn_hist_freight_report_bytes = corn_hist_freight.encode("utf-8")

# test_3=['florian.girardi-ext@ldc.com','gonzalo.lascombes@ldc.com','valentin.chiesa@ldc.com']
# test_2=['florian.girardi-ext@ldc.com']
# # Send email with embedded chart and table
# LDCDataAccessLayerPy.mail.mail_send(
#     to=test_2,
#     subject=f'CORN UPR PRICES APRIL AND JULY    {start_date.strftime("%d-%m")}',
#     from_addr="florian.girardi-ext@ldc.com",
#     mime_type="html",
#     body='Attached the report',
#     attachment={"corn_apr_jul_upr.html": corn_hist_freight_report_bytes}
# )


### GONZA DEMAND

In [0]:


FBV_corn= zema.get_curve(curve="P-CASH-LDC-CALC-PREMIUM-CORN-US-FOB-FBV-CGF-YELLOW-NEW CROP-USDc-Bu", period=f"{start_date}::{end_date}")
FBV_corn=FBV_corn[FBV_corn['observation']=='Last']
FBV_corn_may=FBV_corn[FBV_corn['contract_month']==5]
FBV_corn_apr=FBV_corn[FBV_corn['contract_month']==4]


# sp_mgr.save_pd_to_excel('/sites/grainsargprojects/models/zema_prices/CORN/FBV_corn_may.xlsx',FBV_corn_may,index=False)
# sp_mgr.save_pd_to_excel('/sites/grainsargprojects/models/zema_prices/CORN/FBV_corn_apr.xlsx',FBV_corn_may,index=False)


In [0]:
import plotly.express as px
from datetime import datetime

# Assuming `FBV_corn`, `FBV_corn_may`, and `FBV_corn_apr` are already defined as in your code

# Plot for May contract
fig_may = px.line(
    FBV_corn_may,
    x="date",
    y="value",  # Assuming the price is stored in a column named 'value'
    title="FBV Corn - May Contract",
    labels={"value": "Price (USDc/Bu)", "date": "Date"}
)

# Plot for April contract
fig_apr = px.line(
    FBV_corn_apr,
    x="date",
    y="value",
    title="FBV Corn - April Contract",
    labels={"value": "Price (USDc/Bu)", "date": "Date"}
)

# Show plots
fig_may.show()
fig_apr.show()


In [0]:
cbot_corn= zema.get_curve(curve="P-FUTURE-CBOT-INPUT-CORN-USDc-BU", period=f"{start_date}::{end_date}")
cbot_corn=cbot_corn[cbot_corn['observation']=='Settle']
cbot_corn=cbot_corn[['date','value','contract_year','contract_month']]

In [0]:
cbot_corn_may=cbot_corn[cbot_corn['contract_month']==7]
cbot_corn_may=cbot_corn_may[cbot_corn_may['contract_year']==2025]


In [0]:
# sp_mgr.save_pd_to_excel('/sites/grainsargprojects/models/zema_prices/CORN/corn_cbot_july.xlsx',cbot_corn_may,index=False)

In [0]:
import plotly.express as px
from datetime import datetime

# Assuming `FBV_corn`, `FBV_corn_may`, and `FBV_corn_apr` are already defined as in your code

# Plot for May contract
cbot_may = px.line(
    cbot_corn_may,
    x="date",
    y="value",  # Assuming the price is stored in a column named 'value'
    title="CBOT Corn - May Contract",
    labels={"value": "Price (USDc/Bu)", "date": "Date"}
)
cbot_may.show()

In [0]:
corn_matba= zema.get_curve(curve="P-FUTURE-MATBA-INPUT-CORN-USD-MT", period=f"{start_date}::{end_date}")
corn_matba=corn_matba[corn_matba['observation']=='Settle']

corn_matba=corn_matba[['date','value','contract_year','contract_month']]
corn_matba['day']=corn_matba['date'].dt.day
corn_matba['month']=corn_matba['date'].dt.month
corn_matba['year']=corn_matba['date'].dt.year
corn_matba['virtual_date'] = pd.to_datetime(
    {'year': 2000, 'month': corn_matba['month'], 'day': corn_matba['day']},
    errors='coerce'  # This will convert invalid dates (e.g. Feb 30) to NaT
)
# Drop rows with invalid virtual dates
corn_matba = corn_matba.dropna(subset=['virtual_date'])

# You can now sort or use this date for seasonal charts
corn_matba = corn_matba.sort_values(by='virtual_date')
corn_matba=corn_matba[['date','value','contract_year','contract_month','virtual_date']]
corn_matba['contract_date'] = (
    pd.to_datetime(dict(year=corn_matba['contract_year'],
                        month=corn_matba['contract_month'],
                        day=1))  # temporary first day of month
    + pd.offsets.MonthEnd(0)   # shift to last day of that month
)

In [0]:
sp_mgr.save_pd_to_excel('/sites/grainsargprojects/models/zema_prices/CORN/corn_matba_check.xlsx',corn_matba,index=False)

### MATBA SPREADS

In [0]:

# Split April and July contracts
apr = corn_matba[corn_matba["contract_month"] == 4].copy()
jul = corn_matba[corn_matba["contract_month"] == 7].copy()

# For each trading date, find nearest Apr contract_date >= date
apr_match = (apr.groupby("date")
                .apply(lambda x: x.loc[x["contract_date"] >= x.name].sort_values("contract_date").head(1))
                .reset_index(drop=True))

jul_match = (jul.groupby("date")
                .apply(lambda x: x.loc[x["contract_date"] >= x.name].sort_values("contract_date").head(1))
                .reset_index(drop=True))

# Merge Apr + Jul
spread_df_apr_july = pd.merge(
    apr_match[["date", "value", "contract_date"]].rename(columns={"value": "apr_value", "contract_date": "apr_contract"}),
    jul_match[["date", "value", "contract_date"]].rename(columns={"value": "jul_value", "contract_date": "jul_contract"}),
    on="date"
)

# Compute spread
spread_df_apr_july["spread"] = spread_df_apr_july["apr_value"] - spread_df_apr_july["jul_value"]
spread_df_apr_july['day']=spread_df_apr_july['date'].dt.day
spread_df_apr_july['month']=spread_df_apr_july['date'].dt.month
spread_df_apr_july['year']=spread_df_apr_july['date'].dt.year
spread_df_apr_july['virtual_date'] = pd.to_datetime(
    {'year': 2000, 'month': spread_df_apr_july['month'], 'day': spread_df_apr_july['day']},
    errors='coerce'  # This will convert invalid dates (e.g. Feb 30) to NaT
)

spread_df_apr_july = spread_df_apr_july[
    ((spread_df_apr_july['month'] >= 10) & (spread_df_apr_july['month'] <= 12)) |
    (spread_df_apr_july['month'] <= 4)
]

# Adjust virtual_date: subtract 1 year for Oct–Dec
spread_df_apr_july.loc[spread_df_apr_july['month'] >= 10, 'virtual_date'] = (
    spread_df_apr_july.loc[spread_df_apr_july['month'] >= 10, 'virtual_date']
    - pd.DateOffset(years=1)
)

spread_df_apr_july['season'] = np.where(spread_df_apr_july['month'] >= 10, spread_df_apr_july['year'] + 1,spread_df_apr_july['year'])

# Drop rows with invalid virtual dates
spread_df_apr_july = spread_df_apr_july.dropna(subset=['virtual_date'])
spread_df_apr_july

In [0]:
spread_apr_july = px.line(
    spread_df_apr_july,
    x='virtual_date',
    y='spread',
    color='season',
    labels={
        'virtual_date': '',
        'spread': 'Apr - July Spread',
        'season': 'season'
    },
    title='CORN MATBA JULY - APR',
)

spread_apr_july.update_traces(hovertemplate='%{x|%d-%b}<br>%{y:.2f}<extra>%{fullData.name}</extra>')
spread_apr_july.update_layout(
    xaxis=dict(
        tickformat='%b',  # Month format like Jan, Feb...
        dtick="M1",
        hoverformat='%d-%b'
    ),
    yaxis_title='Price',
    template='plotly_white',
    height=700,
    width=1800,
    hovermode='x unified'
    
)

spread_apr_july.show()


spread_apr_july_html = spread_apr_july.to_html(include_plotlyjs='cdn', full_html=True)

# sp_mgr.save_pd_to_excel('/sites/grainsargprojects/models/zema_prices/CORN/corn_matba_dec.xlsx',corn_matba_DEC,index=False)


In [0]:

# Split April and July contracts
dec = corn_matba[corn_matba["contract_month"] == 12].copy()
jul = corn_matba[corn_matba["contract_month"] == 7].copy()

# For each trading date, find nearest dec contract_date >= date
dec_match = (dec.groupby("date")
                .apply(lambda x: x.loc[x["contract_date"] >= x.name].sort_values("contract_date").head(1))
                .reset_index(drop=True))

jul_match = (jul.groupby("date")
                .apply(lambda x: x.loc[x["contract_date"] >= x.name].sort_values("contract_date").head(1))
                .reset_index(drop=True))

# Merge Apr + Jul
spread_df_july_dec = pd.merge(
    dec_match[["date", "value", "contract_date"]].rename(columns={"value": "dec_value", "contract_date": "dec_contract"}),
    jul_match[["date", "value", "contract_date"]].rename(columns={"value": "jul_value", "contract_date": "jul_contract"}),
    on="date"
)

# Remove rows where jul_value or dec_value is NaN or 0
spread_df_july_dec = spread_df_july_dec.dropna(subset=["jul_value", "dec_value"])
spread_df_july_dec = spread_df_july_dec[(spread_df_july_dec["jul_value"] != 0) & (spread_df_july_dec["dec_value"] != 0)]


# Compute spread
spread_df_july_dec["spread"] = spread_df_july_dec["jul_value"] - spread_df_july_dec["dec_value"] 
spread_df_july_dec['day']=spread_df_july_dec['date'].dt.day
spread_df_july_dec['month']=spread_df_july_dec['date'].dt.month
spread_df_july_dec['year']=spread_df_july_dec['date'].dt.year
spread_df_july_dec['virtual_date'] = pd.to_datetime(
    {'year': 2000, 'month': spread_df_july_dec['month'], 'day': spread_df_july_dec['day']},
    errors='coerce'  # This will convert invalid dates (e.g. Feb 30) to NaT
)

spread_df_july_dec = spread_df_july_dec[
    ((spread_df_july_dec['month'] >= 1) & (spread_df_july_dec['month'] <= 7))]

# Drop rows with invalid virtual dates
spread_df_july_dec = spread_df_july_dec.dropna(subset=['virtual_date'])
spread_df_july_dec


In [0]:
spread_july_dec = px.line(
    spread_df_july_dec,
    x='virtual_date',
    y='spread',
    color='year',
    labels={
        'virtual_date': '',
        'spread': 'July - Dec Spread',
        'year': 'Season'
    },
    title='CORN MATBA July - Dec',
)

spread_july_dec.update_traces(hovertemplate='%{x|%d-%b}<br>%{y:.2f}<extra>%{fullData.name}</extra>')
spread_july_dec.update_layout(
    xaxis=dict(
        tickformat='%b',  # Month format like Jan, Feb...
        dtick="M1",
        hoverformat='%d-%b'
    ),
    yaxis_title='Price',
    template='plotly_white',
    height=700,
    width=1800,
    hovermode='x unified'
    
)

spread_july_dec.show()


spread_july_dec_html = spread_july_dec.to_html(include_plotlyjs='cdn', full_html=True)

# sp_mgr.save_pd_to_excel('/sites/grainsargprojects/models/zema_prices/CORN/corn_matba_dec.xlsx',corn_matba_DEC,index=False)


### MATBA DEC


In [0]:
corn_matba_DEC=corn_matba[corn_matba['contract_month']==12]

corn_matba_DEC['Season'] = 'Dec'+ corn_matba_DEC['contract_year'].astype(str)
corn_matba_DEC = corn_matba_DEC[corn_matba_DEC['date'].dt.year == corn_matba_DEC['contract_year']]
corn_matba_DEC = corn_matba_DEC[corn_matba_DEC['value'] != 0]

In [0]:
dec_matba = px.line(
    corn_matba_DEC,
    x='virtual_date',
    y='value',
    color='Season',
    labels={
        'virtual_date': '',
        'value': 'Spot Price',
        'season': 'Year'
    },
    title='Dec MATBA contract',
)

dec_matba.update_traces(hovertemplate='%{x|%d-%b}<br>%{y:.2f}<extra>%{fullData.name}</extra>')
dec_matba.update_layout(
    xaxis=dict(
        tickformat='%b',  # Month format like Jan, Feb...
        dtick="M1",
        hoverformat='%d-%b'
    ),
    yaxis_title='Price',
    template='plotly_white',
    height=700,
    width=1800,
    hovermode='x unified'
    
)

# Customize specific seasons
for trace in dec_matba.data:
    if trace.name == 'Dec2025':
        trace.line.width = 4
        trace.line.color = 'red'
        trace.line.dash = 'dash'
    elif trace.name == 'Dec2024':
        trace.line.width = 4
        trace.line.color = 'black'
        trace.line.dash = 'solid'

dec_matba.show()


dec_matba_html = dec_matba.to_html(include_plotlyjs='cdn', full_html=True)

# sp_mgr.save_pd_to_excel('/sites/grainsargprojects/models/zema_prices/CORN/corn_matba_dec.xlsx',corn_matba_DEC,index=False)


### MATBA APR

In [0]:
corn_matba_APR=corn_matba[corn_matba['contract_month']==4]

corn_matba_APR['Season'] = 'Apr'+ corn_matba_APR['contract_year'].astype(str)
  
def adjust_virtual_date_corn_apr(row):
    if row['virtual_date'].month in [1, 2,3,4]:
        # Subtract 1 year from the year if month is November or December
        return row['virtual_date'] + pd.DateOffset(years=1)
    else:
        # Keep the date as is if the month is not November or December
        return row['virtual_date'].replace(year=row['virtual_date'].year)
      
corn_matba_APR['virtual_date'] = corn_matba_APR.apply(adjust_virtual_date_corn_apr, axis=1)
corn_matba_APR = corn_matba_APR[corn_matba_APR['value'] != 0]

corn_matba_APR = corn_matba_APR[
    (corn_matba_APR['date'].dt.month.isin([1, 2, 3, 4]) & (corn_matba_APR['date'].dt.year == corn_matba_APR['contract_year'])) |
    (~corn_matba_APR['date'].dt.month.isin([1, 2, 3, 4]) & (corn_matba_APR['date'].dt.year +1 == corn_matba_APR['contract_year']))
]
corn_matba_APR = corn_matba_APR.sort_values(by='date')


apr_matba = px.line(
    corn_matba_APR,
    x='virtual_date',
    y='value',
    color='Season',
    labels={
        'virtual_date': '',
        'value': 'Spot Price',
        'season': 'Year'
    },
    title='Apr MATBA contract',
)

apr_matba.update_traces(hovertemplate='%{x|%d-%b}<br>%{y:.2f}<extra>%{fullData.name}</extra>')
apr_matba.update_layout(
    xaxis=dict(
        tickformat='%b',  # Month format like Jan, Feb...
        dtick="M1",
        hoverformat='%d-%b'
    ),
    yaxis_title='Price',
    template='plotly_white',
    height=700,
    width=1800,
    hovermode='x unified'
    
)

# Customize specific seasons
for trace in apr_matba.data:
    if trace.name == 'Apr2026':
        trace.line.width = 4
        trace.line.color = 'red'
        trace.line.dash = 'dash'
    elif trace.name == 'Apr2025':
        trace.line.width = 4
        trace.line.color = 'black'
        trace.line.dash = 'solid'

apr_matba.show()


apr_matba_html = apr_matba.to_html(include_plotlyjs='cdn', full_html=True)

# sp_mgr.save_pd_to_excel('/sites/grainsargprojects/models/zema_prices/CORN/corn_matba_dec.xlsx',corn_matba_DEC,index=False)


### MATBA JULY

In [0]:
corn_matba_JUL=corn_matba[corn_matba['contract_month']==7]

corn_matba_JUL['Season'] = 'JUL'+ corn_matba_JUL['contract_year'].astype(str)
  
def adjust_virtual_date_corn_JUL(row):
    if row['virtual_date'].month in [1, 2,3,4,5,6,7]:
        # Subtract 1 year from the year if month is November or December
        return row['virtual_date'] + pd.DateOffset(years=1)
    else:
        # Keep the date as is if the month is not November or December
        return row['virtual_date'].replace(year=row['virtual_date'].year)
      
corn_matba_JUL['virtual_date'] = corn_matba_JUL.apply(adjust_virtual_date_corn_JUL, axis=1)
corn_matba_JUL = corn_matba_JUL[corn_matba_JUL['value'] != 0]

corn_matba_JUL = corn_matba_JUL[
    (corn_matba_JUL['date'].dt.month.isin([1, 2, 3, 4,5,6,7]) & (corn_matba_JUL['date'].dt.year == corn_matba_JUL['contract_year'])) |
    (~corn_matba_JUL['date'].dt.month.isin([1, 2, 3, 4,5,6,7]) & (corn_matba_JUL['date'].dt.year +1 == corn_matba_JUL['contract_year']))
]
corn_matba_JUL = corn_matba_JUL.sort_values(by='date')


JUL_matba = px.line(
    corn_matba_JUL,
    x='virtual_date',
    y='value',
    color='Season',
    labels={
        'virtual_date': '',
        'value': 'Spot Price',
        'season': 'Year'
    },
    title='JUL MATBA contract',
)

JUL_matba.update_traces(hovertemplate='%{x|%d-%b}<br>%{y:.2f}<extra>%{fullData.name}</extra>')
JUL_matba.update_layout(
    xaxis=dict(
        tickformat='%b',  # Month format like Jan, Feb...
        dtick="M1",
        hoverformat='%d-%b'
    ),
    yaxis_title='Price',
    template='plotly_white',
    height=700,
    width=1800,
    hovermode='x unified'
    
)

# Customize specific seasons
for trace in JUL_matba.data:
    if trace.name == 'JUL2026':
        trace.line.width = 4
        trace.line.color = 'red'
        trace.line.dash = 'dash'
    elif trace.name == 'JUL2025':
        trace.line.width = 4
        trace.line.color = 'black'
        trace.line.dash = 'solid'

JUL_matba.show()


JUL_matba_html = JUL_matba.to_html(include_plotlyjs='cdn', full_html=True)

# sp_mgr.save_pd_to_excel('/sites/grainsargprojects/models/zema_prices/CORN/corn_matba_dec.xlsx',corn_matba_DEC,index=False)


In [0]:
corn_upr= zema.get_curve(curve="P-CASH-LDC-INPUT-PREMIUM-CORN-AR-FOB Up River Arg-USDc-Bu", period=f"{start_date}::{end_date}")
corn_upr=corn_upr[corn_upr['observation']=='Last']

corn_bb= zema.get_curve(curve="P-CASH-LDC-INPUT-PREMIUM-CORN-AR-FOB Bahia Blanca Arg-USDc-Bu", period=f"{start_date}::{end_date}")
corn_bb=corn_bb[corn_bb['observation']=='Last']


corn_upr['day']=corn_upr['date'].dt.day
corn_upr['month']=corn_upr['date'].dt.month
corn_upr['year']=corn_upr['date'].dt.year


corn_bb['day']=corn_bb['date'].dt.day
corn_bb['month']=corn_bb['date'].dt.month
corn_bb['year']=corn_bb['date'].dt.year

def get_spot_contract(row):
    if row['month'] == 12:
        contract_month = 1
        contract_year = row['year'] + 1
    else:
        contract_month = row['month'] + 1
        contract_year = row['year']
    return pd.Series({'target_month': contract_month, 'target_year': contract_year})

# Apply to get target contract for each date
corn_bb[['target_month', 'target_year']] = corn_bb.apply(get_spot_contract, axis=1)
corn_upr[['target_month', 'target_year']] = corn_upr.apply(get_spot_contract, axis=1)



fob_bb = corn_bb[
    (corn_bb['contract_month'] == corn_bb['target_month']) &
    (corn_bb['contract_year'] == corn_bb['target_year'])
].drop(columns=['month', 'year', 'target_month', 'target_year'])
fob_bb = fob_bb.rename(columns={"value": "prem_bb"})
fob_bb=fob_bb[['date','prem_bb']]

# Filter spot prems
fob_upr = corn_upr[
    (corn_upr['contract_month'] == corn_upr['target_month']) &
    (corn_upr['contract_year'] == corn_upr['target_year'])
].drop(columns=['month', 'year', 'target_month', 'target_year'])
fob_upr = fob_upr.rename(columns={"value": "prem_upr"})
fob_upr=fob_upr[['date','prem_upr']]


corn_spot_upr_bb=pd.merge(fob_bb,fob_upr,on='date')
corn_spot_upr_bb['Premium']=corn_spot_upr_bb['prem_bb']-corn_spot_upr_bb['prem_upr']

corn_spot_upr_bb['season']= corn_spot_upr_bb.apply(assign_season_corn, axis=1)
corn_spot_upr_bb['date'] = pd.to_datetime(corn_spot_upr_bb['date'])
corn_spot_upr_bb['virtual_date'] = pd.to_datetime({'year': 2000, 'month': corn_spot_upr_bb['date'].dt.month, 'day': corn_spot_upr_bb['date'].dt.day},errors='coerce')
corn_spot_upr_bb['virtual_date'] = corn_spot_upr_bb.apply(adjust_virtual_date_corn, axis=1)
corn_spot_upr_bb

# sp_mgr.save_pd_to_excel('/sites/grainsargprojects/models/zema_prices/CORN/corn_spot_BB_vs_upr.xlsx',corn_spot_upr_bb,index=False)









## SPOT

In [0]:
import pandas as pd
import plotly.express as px

# Ensure datetime parsing
arg_corn['date'] = pd.to_datetime(arg_corn['date'])
arg_corn['virtual_date'] = pd.to_datetime(arg_corn['virtual_date'])

# Create contract_date to identify the earliest available contract
arg_corn['contract_date'] = pd.to_datetime(dict(year=arg_corn['contract_year'], month=arg_corn['contract_month'], day=1))

# For each real date, get the earliest contract
spot_corn_df = arg_corn.sort_values(['date', 'contract_date']).groupby('date').first().reset_index()

# Add a season label (year of the real date)
spot_corn_df['season'] = spot_corn_df['date'].dt.year
spot_corn_df = spot_corn_df[spot_corn_df['season'] != 2020]

# Plot with Plotly
spot_corn = px.line(
    spot_corn_df,
    x='virtual_date',
    y='value',
    color='season',
    labels={
        'virtual_date': '',
        'value': 'Spot Price',
        'season': 'Year'
    },
    title='Corn UPR Spot Price'
)

spot_corn.update_traces(hovertemplate='%{x|%d-%b}<br>%{y:.2f}<extra>%{fullData.name}</extra>')

spot_corn.update_layout(
    xaxis=dict(
        tickformat='%b',  # Month format like Jan, Feb...
        dtick="M1",
        hoverformat='%d-%b'
    ),
    yaxis_title='Price',
    template='plotly_white',
    height=700,
    width=1800,
    hovermode='x unified'  
)

spot_corn.show()


spot_corn_html = spot_corn.to_html(include_plotlyjs='cdn', full_html=True)



In [0]:
# spot_corn_df_gonza=spot_corn_df[['date','value','season']]
# sp_mgr.save_pd_to_excel('/sites/grainsargprojects/models/zema_prices/CORN/corn_spot_upr.xlsx',spot_corn_df,index=False)

# BARLEY

In [0]:
start_date = datetime(2018, 1, 1)

end_date = datetime(2030, 1, 1)

## BARLEY VS MATIF

In [0]:
eurusd

In [0]:
def assign_season_barley(row):
    date = row['date']
    if date.month >= 11:  # March to December → same year
        season_start = date.year
    else:  # January, February → previous year's marketing season
        season_start = date.year - 1
    return f"{season_start}/{season_start + 1}"

def adjust_virtual_date(row):
    if row['virtual_date'].month in [11, 12]:
        # Subtract 1 year from the year if month is November or December
        return row['virtual_date'].replace(year=row['virtual_date'].year - 1)
    else:
        # Keep the date as is if the month is not November or December
        return row['virtual_date']

In [0]:

matif= zema.get_curve(curve="P-FUTURE-ENXT-INPUT-WHEAT-EUR-MT", period=f"{start_date}::{end_date}")

matif=matif[matif['observation']=='Settle']
matif.rename(columns={'value': 'matif_eur'}, inplace=True)
matif=matif[['date','matif_eur','contract_year','contract_month']]

merged_df = pd.merge(
    matif,
    eurusd,
    on=['date', 'contract_year', 'contract_month'],
    how='inner'  # You can also use 'outer', 'left', or 'right' depending on your needs
)
merged_df['matif_usd']=merged_df['matif_eur']*merged_df['eurusd']

merged_df=merged_df[['date','matif_usd','contract_year','contract_month']]

matif_raw=merged_df.copy()
# Make sure date is datetime

matif_raw['date'] = pd.to_datetime(matif_raw['date'])

# Create a reference year column based on the date
matif_raw['ref_contract_year'] = matif_raw['date'].apply(lambda x: x.year + 1 if x.month >= 3 else x.year)

# Filter to March contracts matching the reference year
matif_march = matif_raw[
    (matif_raw['contract_month'] == 3) &
    (matif_raw['contract_year'] == matif_raw['ref_contract_year'])
].copy()

# Drop the helper column if needed
matif_march.drop(columns='ref_contract_year', inplace=True)

matif_march['date'] = pd.to_datetime(matif_march['date'])

# # Determine the correct contract year for each date
# matif_march['target_contract_year'] = matif_march['date'].dt.year
# matif_march.loc[matif_march['date'].dt.month >= 3, 'target_contract_year'] += 1

# # Filter to only keep rows where contract_year == target_contract_year
# matif_march_closest_contract = matif_march[matif_march['contract_year'] == matif_march['target_contract_year']].copy()


fob_barley = zema.get_curve(curve='T-CASH-LDC-RISK-FLAT-BARLEY-AR-FOB Bahia Blanca-USD-MT', period=f"{start_date}::{end_date}")
fob_barley=fob_barley[['date','value']]
barley_march_matif=pd.merge(fob_barley,matif_march,on='date')
barley_march_matif['Premium']=barley_march_matif['value']-barley_march_matif['matif_usd']


barley_march_matif['season']= barley_march_matif.apply(assign_season_barley, axis=1)
barley_march_matif['date'] = pd.to_datetime(barley_march_matif['date'])
barley_march_matif['virtual_date'] = pd.to_datetime({'year': 2000, 'month': barley_march_matif['date'].dt.month, 'day': barley_march_matif['date'].dt.day},errors='coerce')
barley_march_matif['virtual_date'] = barley_march_matif.apply(adjust_virtual_date, axis=1)

# Plot with Plotly
prem_barley_matif = px.line(
    barley_march_matif,
    x='virtual_date',
    y='Premium',
    color='season',
    labels={
        'virtual_date': '',
        'Premium': 'Premium',
        'season': 'Marketing Year'
    },
    title='ARG FOB BARLEY - MARCH MATIF'
)

prem_barley_matif.update_traces(hovertemplate='%{x|%d-%b}<br>%{y:.2f}<extra>%{fullData.name}</extra>')
prem_barley_matif.update_layout(
    xaxis=dict(
        tickformat='%b',  # Month format like Jan, Feb...
        dtick="M1",
        hoverformat='%d-%b'
    ),
    yaxis_title='Premium',
    template='plotly_white',
    height=700,
    width=1800,
    hovermode='x unified'
)


prem_barley_matif_html = prem_barley_matif.to_html(include_plotlyjs='cdn', full_html=True)



### MATIF DEC

In [0]:
# matif_raw['ref_contract_year'] = matif_raw['date'].apply(lambda x: x.year + 1 if x.month >= 12 else x.year)
matif_raw['ref_contract_year'] = matif_raw['date'].apply(lambda x: x.year)

# Filter to March contracts matching the reference year
matif_dec = matif_raw[
    (matif_raw['contract_month'] == 12) &
    (matif_raw['contract_year'] == matif_raw['ref_contract_year'])
].copy()

# Drop the helper column if needed
matif_dec.drop(columns='ref_contract_year', inplace=True)

matif_dec['date'] = pd.to_datetime(matif_dec['date'])

fob_barley = zema.get_curve(curve='T-CASH-LDC-RISK-FLAT-BARLEY-AR-FOB Bahia Blanca-USD-MT', period=f"{start_date}::{end_date}")
fob_barley=fob_barley[['date','value']]
barley_dec_matif=pd.merge(fob_barley,matif_dec,on='date')

barley_dec_matif['Premium']=barley_dec_matif['value']-barley_dec_matif['matif_usd']


barley_dec_matif['season']= barley_dec_matif.apply(assign_season_barley, axis=1)
barley_dec_matif['date'] = pd.to_datetime(barley_dec_matif['date'])
barley_dec_matif['virtual_date'] = pd.to_datetime({'year': 2000, 'month': barley_dec_matif['date'].dt.month, 'day': barley_dec_matif['date'].dt.day},errors='coerce')
barley_dec_matif['virtual_date'] = barley_dec_matif.apply(adjust_virtual_date, axis=1)

# Plot with Plotly
prem_barley_matif_dec = px.line(
    barley_dec_matif,
    x='virtual_date',
    y='Premium',
    color='season',
    labels={
        'virtual_date': '',
        'Premium': 'Premium',
        'season': 'Marketing Year'
    },
    title='ARG FOB BARLEY - DEC MATIF'
)

prem_barley_matif_dec.update_traces(hovertemplate='%{x|%d-%b}<br>%{y:.2f}<extra>%{fullData.name}</extra>')
prem_barley_matif_dec.update_layout(
    xaxis=dict(
        tickformat='%b',  # Month format like Jan, Feb...
        dtick="M1",
        hoverformat='%d-%b'
    ),
    yaxis_title='Premium',
    template='plotly_white',
    height=700,
    width=1800,
    hovermode='x unified'
)

prem_barley_matif_dec.show()

prem_barley_matif_dec_html = prem_barley_matif_dec.to_html(include_plotlyjs='cdn', full_html=True)



### WHEAT UPR

In [0]:
wheat = zema.get_curve(curve="P-CASH-LDC-INPUT-FLAT-WHEAT-AR-FOB Up River-11.5-USD-MT", period=f"{start_date}::{end_date}")
wheat=wheat[wheat['observation']=='Last']
wheat=wheat[['date','value','contract_year','contract_month']]
wheat['day']=wheat['date'].dt.day
wheat['month']=wheat['date'].dt.month
wheat['year']=wheat['date'].dt.year
wheat['virtual_date'] = pd.to_datetime(
    {'year': 2000, 'month': wheat['month'], 'day': wheat['day']},
    errors='coerce'  # This will convert invalid dates (e.g. Feb 30) to NaT
)

# Drop rows with invalid virtual dates
wheat = wheat.dropna(subset=['virtual_date'])

# You can now sort or use this date for seasonal charts
wheat = wheat.sort_values(by='virtual_date')
wheat=wheat[['date','value','contract_year','contract_month','virtual_date']]

wheat_spot=wheat.copy()
# Ensure datetime format
wheat_spot['date'] = pd.to_datetime(wheat_spot['date'])

# Extract current month and year from the 'date' column
wheat_spot['ref_month'] = wheat_spot['date'].dt.month
wheat_spot['ref_year'] = wheat_spot['date'].dt.year

# Filter to potential spot contracts
spot_candidates = wheat_spot[
    (wheat_spot['contract_year'] == wheat_spot['ref_year']) &
    (wheat_spot['contract_month'] == wheat_spot['ref_month'])
].copy()

# Keep the first occurrence of the contract per date
spot_contracts = (
    spot_candidates
    .sort_values(['date'])  # Make sure earliest stays first
    .drop_duplicates(subset='date', keep='first')
)

# Optionally keep only relevant columns
UPR_Wheat= spot_contracts[['date', 'value']]
UPR_Wheat.rename(columns={'value': 'fob_upr'}, inplace=True)

fob_upr_wheat_barley=pd.merge(UPR_Wheat,fob_barley,on='date')
fob_upr_wheat_barley['Premium']=fob_upr_wheat_barley['value']-fob_upr_wheat_barley['fob_upr']

fob_upr_wheat_barley['season']= fob_upr_wheat_barley.apply(assign_season_barley, axis=1)
fob_upr_wheat_barley['date'] = pd.to_datetime(fob_upr_wheat_barley['date'])
fob_upr_wheat_barley['virtual_date'] = pd.to_datetime({'year': 2000, 'month': fob_upr_wheat_barley['date'].dt.month, 'day': fob_upr_wheat_barley['date'].dt.day},errors='coerce')
fob_upr_wheat_barley['virtual_date'] = fob_upr_wheat_barley.apply(adjust_virtual_date, axis=1)

# Plot with Plotly
upr_w_vs_barley = px.line(
    fob_upr_wheat_barley,
    x='virtual_date',
    y='Premium',
    color='season',
    labels={
        'virtual_date': '',
        'Premium': 'Premium',
        'season': 'Marketing Year'
    },
    title='ARG FOB BARLEY - FOB WHEAT UPR'
)

upr_w_vs_barley.update_traces(hovertemplate='%{x|%d-%b}<br>%{y:.2f}<extra>%{fullData.name}</extra>')
upr_w_vs_barley.update_layout(
    xaxis=dict(
        tickformat='%b',  # Month format like Jan, Feb...
        dtick="M1",
        hoverformat='%d-%b'
    ),
    yaxis_title='Premium',
    template='plotly_white',
    height=700,
    width=1800,
    hovermode='x unified'
)

upr_w_vs_barley.show()

upr_w_vs_barley_html = upr_w_vs_barley.to_html(include_plotlyjs='cdn', full_html=True)



### CORN BB

In [0]:
corn_bb = zema.get_curve(curve="P-CASH-LDC-CALC-FLAT-CORN-AR-FOB Bahia Blanca Arg-USD-MT", period=f"{start_date}::{end_date}")
corn_bb=corn_bb[corn_bb['observation']=='Last']
corn_bb=corn_bb[['date','value','contract_year','contract_month']]
corn_bb['day']=corn_bb['date'].dt.day
corn_bb['month']=corn_bb['date'].dt.month
corn_bb['year']=corn_bb['date'].dt.year
corn_bb['virtual_date'] = pd.to_datetime(
    {'year': 2000, 'month': corn_bb['month'], 'day': corn_bb['day']},
    errors='coerce'  # This will convert invalid dates (e.g. Feb 30) to NaT
)


# Drop rows with invalid virtual dates
corn_bb = corn_bb.dropna(subset=['virtual_date'])

# You can now sort or use this date for seasonal charts
corn_bb = corn_bb.sort_values(by='virtual_date')
corn_bb=corn_bb[['date','value','contract_year','contract_month','virtual_date']]

corn_bb_spot=corn_bb.copy()
# Ensure datetime format
corn_bb_spot['date'] = pd.to_datetime(corn_bb_spot['date'])

# Extract current month and year from the 'date' column
corn_bb_spot['ref_month'] = corn_bb_spot['date'].dt.month
corn_bb_spot['ref_year'] = corn_bb_spot['date'].dt.year

# Filter to potential spot contracts
spot_candidates = corn_bb_spot[
    (corn_bb_spot['contract_year'] == corn_bb_spot['ref_year']) &
    (corn_bb_spot['contract_month'] == corn_bb_spot['ref_month'])
].copy()

# Keep the first occurrence of the contract per date
spot_contracts = (
    spot_candidates
    .sort_values(['date'])  # Make sure earliest stays first
    .drop_duplicates(subset='date', keep='first')
)

# Optionally keep only relevant columns
corn_bb= spot_contracts[['date', 'value']]
corn_bb.rename(columns={'value': 'corn_bb'}, inplace=True)

fob_corn_bb_barley=pd.merge(corn_bb,fob_barley,on='date')
fob_corn_bb_barley['Premium']=fob_corn_bb_barley['value']-fob_corn_bb_barley['corn_bb']

fob_corn_bb_barley['season']= fob_corn_bb_barley.apply(assign_season_barley, axis=1)
fob_corn_bb_barley['date'] = pd.to_datetime(fob_corn_bb_barley['date'])
fob_corn_bb_barley['virtual_date'] = pd.to_datetime({'year': 2000, 'month': fob_corn_bb_barley['date'].dt.month, 'day': fob_corn_bb_barley['date'].dt.day},errors='coerce')
fob_corn_bb_barley['virtual_date'] = fob_corn_bb_barley.apply(adjust_virtual_date, axis=1)

# Plot with Plotly
BB_c_vs_barley = px.line(
    fob_corn_bb_barley,
    x='virtual_date',
    y='Premium',
    color='season',
    labels={
        'virtual_date': '',
        'Premium': 'Premium',
        'season': 'Marketing Year'
    },
    title='ARG FOB BARLEY - BB CORN UPR'
)

BB_c_vs_barley.update_traces(hovertemplate='%{x|%d-%b}<br>%{y:.2f}<extra>%{fullData.name}</extra>')
BB_c_vs_barley.update_layout(
    xaxis=dict(
        tickformat='%b',  # Month format like Jan, Feb...
        dtick="M1",
        hoverformat='%d-%b'
    ),
    yaxis_title='Premium',
    template='plotly_white',
    height=700,
    width=1800,
    hovermode='x unified'
)

BB_c_vs_barley.show()

BB_c_vs_barley_html = BB_c_vs_barley.to_html(include_plotlyjs='cdn', full_html=True)



In [0]:
MATIF_VS_BARLEY_html_report  = f"""
<!DOCTYPE html>
<html>
<head>
    <title>Markets Snapshot</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            padding: 20px;
        }}
        h2 {{
            margin-top: 40px;
            color: #2c3e50;
        }}
        .chart-container {{
            margin-bottom: 50px;
        }}
    </style>
</head>
<body>
    <h1>MARCH MATIF VS ARG FOB BARLEY</h1>

    <div class="chart-container">{prem_barley_matif_html}</div>

    <h1>DEC MATIF VS ARG FOB BARLEY</h1>

    <div class="chart-container">{prem_barley_matif_dec_html}</div>

    <h1>WHEAT UPR VS ARG FOB BARLEY</h1>

    <div class="chart-container">{upr_w_vs_barley_html}</div>

    <h1>CORN BB VS ARG FOB BARLEY</h1>

    <div class="chart-container">{BB_c_vs_barley_html}</div>


</body>
</html>
"""

MATIF_VS_BARLEY_report_bytes = MATIF_VS_BARLEY_html_report.encode("utf-8")

In [0]:

# AUS_Barley = zema.get_curve(curve='P-FUTURE-ASX-INPUT-BARLEY-EASTERN-FEED-AUD-MT', period=f"{start_date}::{end_date}")

# AUS_Barley=AUS_Barley[AUS_Barley['observation']=='Settle']
# AUS_Barley_nov=AUS_Barley[AUS_Barley['contract_month']==11]
# AUS_Barley_nov=AUS_Barley_nov[['date','value','contract_month','contract_year']]
# AUS_Barley_nov=AUS_Barley_nov.rename(columns={"value": "price_AUD",'date':'Date'})

# url_aud = 'https://www.rba.gov.au/statistics/tables/xls-hist/2023-current.xls'

# AUD_USD_raw = pd.read_excel(url_aud, sheet_name=0, header=None)

# AUD_USD_raw = pd.read_excel(url_aud, header=None)

# # The actual data starts after row 10
# data = AUD_USD_raw[10:].copy()

# # Set the proper column names from row 5 (Units row)
# headers = AUD_USD_raw.iloc[5].tolist()
# headers[0] = 'Date'  # Rename first column
# data.columns = headers

# # Keep only 'Date' and 'USD' columns (AUD/USD FX rate)
# AUD_USD_fx = data[['Date', 'USD']].copy()

# # Clean data types
# AUD_USD_fx['Date'] = pd.to_datetime(AUD_USD_fx['Date'], errors='coerce')
# AUD_USD_fx['AUDUSD'] = pd.to_numeric(AUD_USD_fx['USD'], errors='coerce')

# # Drop rows with missing values
# AUD_USD_fx = AUD_USD_fx.dropna(subset=['Date', 'AUDUSD'])

# # Drop original 'USD' column
# AUD_USD_fx = AUD_USD_fx[['Date', 'AUDUSD']]

# AUS_Barley_nov=pd.merge(AUS_Barley_nov,AUD_USD_fx,on='Date')
# AUS_Barley_nov['price_USD']=AUS_Barley_nov['price_AUD']*AUS_Barley_nov['AUDUSD']

# # Apply the function to get target December contract for each row
# matif_raw[['target_month', 'target_year']] = matif_raw.apply(get_nearest_december_contract, axis=1)

# # Now filter the DataFrame to keep only the rows that match the target Dec contract
# matif_dec = matif_raw[
#     (matif_raw['contract_month'] == matif_raw['target_month']) &
#     (matif_raw['contract_year'] == matif_raw['target_year'])
# ].copy()

# matif_dec=matif_dec[['date','matif_usd']]


# matif_dec=matif_dec.rename(columns={'date':'Date'})
# AUS_Barley_NOV_MATIF_DEC=pd.merge(AUS_Barley_nov,matif_dec,on='Date')
# AUS_Barley_NOV_MATIF_DEC['Prem']=AUS_Barley_NOV_MATIF_DEC['price_USD']-AUS_Barley_NOV_MATIF_DEC['matif_usd']
# # Plot with Plotly
# AUS_barley_vs_matif = px.line(
#     AUS_Barley_NOV_MATIF_DEC,
#     x='Date',
#     y='Prem',
#     title='AUS NOV BARLEY VS DEC MATIF WHEAT'
# )

# AUS_barley_vs_matif.show()
# AUS_barley_vs_matif_html = AUS_barley_vs_matif.to_html(include_plotlyjs='cdn', full_html=True)


# MATIF_VS_BARLEY_html_report  = f"""
# <!DOCTYPE html>
# <html>
# <head>
#     <title>Markets Snapshot</title>
#     <style>
#         body {{
#             font-family: Arial, sans-serif;
#             padding: 20px;
#         }}
#         h2 {{
#             margin-top: 40px;
#             color: #2c3e50;
#         }}
#         .chart-container {{
#             margin-bottom: 50px;
#         }}
#     </style>
# </head>
# <body>
#     <h1>MARCH MATIF VS ARG FOB BARLEY</h1>

#     <div class="chart-container">{prem_barley_dec_matif}</div>
# </body>
# </html>
# """

# MATIF_VS_BARLEY_report_bytes = MATIF_VS_BARLEY_html_report.encode("utf-8")

# html_content = f"""
# <!DOCTYPE html>
# <html>
# <head>
#     <title>Markets Snapshot</title>
#     <style>
#         body {{
#             font-family: Arial, sans-serif;
#             margin: 20px;
#         }}
#         h1, h2 {{
#             margin-top: 40px;
#             color: #333;
#         }}
#         img {{
#             width: 1800px;
#             max-width: 100%;
#             min-width: 1000px;
#             display: block;
#             margin-bottom: 30px;
#         }}
#     </style>
# </head>
# <body>
#     <h1>MARCH MATIF VS FOB BARLEY</h1>

#     <img src="spot_wheat" alt="Spot">

# </body>
# </html>
# """

# grains=['florian.girardi-ext@ldc.com','Roman.Avramishin@LDC.com','juan.garciafuentes@ldc.com','gonzalo.lascombes@ldc.com','juan.carnemolla@LDC.com','valentin.chiesa@ldc.com','nicolas.benaicha@ldc.com']
# test=['florian.girardi-ext@ldc.com']

# # Send email with embedded chart and table
# LDCDataAccessLayerPy.mail.mail_send(
#     to=test,
#     subject=f'MARCH MATIF VS BARLEY {datetime.now().strftime("%d-%m")}',
#     from_addr="florian.girardi-ext@ldc.com",
#     body=html_content,
#     mime_type="html",
#     html_images={"spot_wheat": prem_barley_dec_matif},
#     attachment={"march_matif_vs_barley_fob.html": MATIF_VS_BARLEY_report_bytes}
# )




#RATIOS MATBA

In [0]:
corn_matba= zema.get_curve(curve="P-FUTURE-MATBA-INPUT-CORN-USD-MT", period=f"{start_date}::{end_date}")
corn_matba=corn_matba[corn_matba['observation']=='Settle']

corn_matba=corn_matba[['date','value','contract_year','contract_month']]
corn_matba['day']=corn_matba['date'].dt.day
corn_matba['month']=corn_matba['date'].dt.month
corn_matba['year']=corn_matba['date'].dt.year
corn_matba['virtual_date'] = pd.to_datetime(
    {'year': 2000, 'month': corn_matba['month'], 'day': corn_matba['day']},
    errors='coerce'  # This will convert invalid dates (e.g. Feb 30) to NaT
)
# Drop rows with invalid virtual dates
corn_matba = corn_matba.dropna(subset=['virtual_date'])

# You can now sort or use this date for seasonal charts
corn_matba = corn_matba.sort_values(by='virtual_date')
corn_matba=corn_matba[['date','value','contract_year','contract_month','virtual_date']]
corn_matba['contract_date'] = (
    pd.to_datetime(dict(year=corn_matba['contract_year'],
                        month=corn_matba['contract_month'],
                        day=1))  # temporary first day of month
    + pd.offsets.MonthEnd(0)   # shift to last day of that month
)
corn_matba_APR=corn_matba[corn_matba['contract_month']==4]

corn_matba_APR['Season'] = 'Apr'+ corn_matba_APR['contract_year'].astype(str)
  
def adjust_virtual_date_corn_apr(row):
    if row['virtual_date'].month in [1, 2,3,4]:
        # Subtract 1 year from the year if month is November or December
        return row['virtual_date'] + pd.DateOffset(years=1)
    else:
        # Keep the date as is if the month is not November or December
        return row['virtual_date'].replace(year=row['virtual_date'].year)
      
corn_matba_APR['virtual_date'] = corn_matba_APR.apply(adjust_virtual_date_corn_apr, axis=1)
corn_matba_APR = corn_matba_APR[corn_matba_APR['value'] != 0]

corn_matba_APR = corn_matba_APR[
    (corn_matba_APR['date'].dt.month.isin([1, 2, 3, 4]) & (corn_matba_APR['date'].dt.year == corn_matba_APR['contract_year'])) |
    (~corn_matba_APR['date'].dt.month.isin([1, 2, 3, 4]) & (corn_matba_APR['date'].dt.year +1 == corn_matba_APR['contract_year']))
]
corn_matba_APR = corn_matba_APR.sort_values(by='date')


apr_matba = px.line(
    corn_matba_APR,
    x='virtual_date',
    y='value',
    color='Season',
    labels={
        'virtual_date': '',
        'value': 'Spot Price',
        'season': 'Year'
    },
    title='Apr MATBA contract',
)

apr_matba.update_traces(hovertemplate='%{x|%d-%b}<br>%{y:.2f}<extra>%{fullData.name}</extra>')
apr_matba.update_layout(
    xaxis=dict(
        tickformat='%b',  # Month format like Jan, Feb...
        dtick="M1",
        hoverformat='%d-%b'
    ),
    yaxis_title='Price',
    template='plotly_white',
    height=700,
    width=1800,
    hovermode='x unified'
    
)

# Customize specific seasons
for trace in apr_matba.data:
    if trace.name == 'Apr2026':
        trace.line.width = 4
        trace.line.color = 'red'
        trace.line.dash = 'dash'
    elif trace.name == 'Apr2025':
        trace.line.width = 4
        trace.line.color = 'black'
        trace.line.dash = 'solid'

apr_matba_html = apr_matba.to_html(include_plotlyjs='cdn', full_html=True)

soy_matba= zema.get_curve(curve="P-FUTURE-MATBA-INPUT-SOYBEAN-USD-MT", period=f"{start_date}::{end_date}")
soy_matba=soy_matba[soy_matba['observation']=='Settle']

soy_matba=soy_matba[['date','value','contract_year','contract_month']]
soy_matba['day']=soy_matba['date'].dt.day
soy_matba['month']=soy_matba['date'].dt.month
soy_matba['year']=soy_matba['date'].dt.year
soy_matba['virtual_date'] = pd.to_datetime(
    {'year': 2000, 'month': soy_matba['month'], 'day': soy_matba['day']},
    errors='coerce'  # This will convert invalid dates (e.g. Feb 30) to NaT
)
# Drop rows with invalid virtual dates
soy_matba = soy_matba.dropna(subset=['virtual_date'])

# You can now sort or use this date for seasonal charts
soy_matba = soy_matba.sort_values(by='virtual_date')
soy_matba=soy_matba[['date','value','contract_year','contract_month','virtual_date']]
soy_matba['contract_date'] = (
    pd.to_datetime(dict(year=soy_matba['contract_year'],
                        month=soy_matba['contract_month'],
                        day=1))  # temporary first day of month
    + pd.offsets.MonthEnd(0)   # shift to last day of that month
)

soy_matba=soy_matba.sort_values(by='date')

may = soy_matba[soy_matba["contract_month"] == 5].copy()
apr=corn_matba[corn_matba["contract_month"] == 4].copy()

apr_match = (apr.groupby("date")
                .apply(lambda x: x.loc[x["contract_date"] >= x.name].sort_values("contract_date").head(1))
                .reset_index(drop=True))

may_match = (may.groupby("date")
                .apply(lambda x: x.loc[x["contract_date"] >= x.name].sort_values("contract_date").head(1))
                .reset_index(drop=True))

may_match.drop(["contract_year", "contract_month", "contract_date","virtual_date"], axis=1, inplace=True)
may_match.rename(columns={"value": "SB"}, inplace=True)


apr_match.drop(["contract_year", "contract_month", "contract_date","virtual_date"], axis=1, inplace=True)
apr_match.rename(columns={"value": "CORN"}, inplace=True)


# Merge Apr + Jul
ratio_df_apr_may = pd.merge(may_match,apr_match, on="date")

ratio_df_apr_may = ratio_df_apr_may[ratio_df_apr_may['CORN'] != 0]

# Compute spread
ratio_df_apr_may["ratio"] = ratio_df_apr_may["SB"]/ratio_df_apr_may["CORN"]
ratio_df_apr_may['day']=ratio_df_apr_may['date'].dt.day
ratio_df_apr_may['month']=ratio_df_apr_may['date'].dt.month
ratio_df_apr_may['year']=ratio_df_apr_may['date'].dt.year
ratio_df_apr_may['virtual_date'] = pd.to_datetime(
    {'year': 2000, 'month': ratio_df_apr_may['month'], 'day': ratio_df_apr_may['day']},
    errors='coerce'  # This will convert invalid dates (e.g. Feb 30) to NaT
)

ratio_df_apr_may = ratio_df_apr_may[
    ((ratio_df_apr_may['month'] >= 6) & (ratio_df_apr_may['month'] <= 12)) |
    (ratio_df_apr_may['month'] <= 4)
]

# Adjust virtual_date: subtract 1 year for Oct–Dec
ratio_df_apr_may.loc[ratio_df_apr_may['month'] >= 6, 'virtual_date'] = (
    ratio_df_apr_may.loc[ratio_df_apr_may['month'] >= 6, 'virtual_date']
    - pd.DateOffset(years=1)
)


ratio_df_apr_may['season'] = np.where(ratio_df_apr_may['month'] >= 6, ratio_df_apr_may['year'] + 1,ratio_df_apr_may['year'])

# Drop rows with invalid virtual dates
ratio_df_apr_may = ratio_df_apr_may.dropna(subset=['virtual_date'])
ratio_df_apr_may

import plotly.express as px

# Create the base plot
ratio_apr_july = px.line(
    ratio_df_apr_may,
    x="virtual_date",
    y="ratio",
    color="season",
    labels={
        "virtual_date": "",
        "ratio": "MATBA Ratios SB MAY/CORN APR ",
        "season": "season"
    },
    title="SB MAY/CORN APR",
)

# Compute average across seasons
avg_df = ratio_df_apr_may.groupby("virtual_date", as_index=False)["ratio"].mean()
ratio_apr_july.add_scatter(
    x=avg_df["virtual_date"],
    y=avg_df["ratio"],
    mode="lines",
    name="Average",
    line=dict(color="gray", width=3, dash="dash")  # dashed gray line
)

# Identify last two seasons
seasons_sorted = sorted(ratio_df_apr_may["season"].unique())
last_season = seasons_sorted[-1]
prev_season = seasons_sorted[-2]

# Update traces: highlight last and previous seasons
for trace in ratio_apr_july.data:
    if trace.name == str(last_season):
        trace.line.color = "red"
        trace.line.dash = "dash"
        trace.line.width = 5
    elif trace.name == str(prev_season):
        trace.line.color = "black"
        trace.line.dash = "solid"
        trace.line.width = 3

# Layout and hover
ratio_apr_july.update_traces(
    hovertemplate="%{x|%d-%b}<br>%{y:.2f}<extra>%{fullData.name}</extra>"
)
ratio_apr_july.update_layout(
    xaxis=dict(
        tickformat="%b",
        dtick="M1",
        hoverformat="%d-%b"
    ),
    yaxis_title="Price",
    template="plotly_white",
    height=700,
    width=1800,
    hovermode="x unified"
)


ratio_apr_july_html = ratio_apr_july.to_html(include_plotlyjs="cdn", full_html=True)

import plotly.graph_objects as go
import plotly.express as px

seasons = ratio_df_apr_may["season"].unique()
line_styles = {
    "ratio": dict(dash="solid", width=2),
    "SB": dict(dash="dot", width=2),
    "CORN": dict(dash="dash", width=2)
}
color_map = {season: px.colors.qualitative.Plotly[i % len(px.colors.qualitative.Plotly)]
             for i, season in enumerate(seasons)}

fig_apr_jul = go.Figure()

# Add all traces
for season in seasons:
    df_season = ratio_df_apr_may[ratio_df_apr_may["season"] == season]
    # ratio
    fig_apr_jul.add_trace(go.Scatter(
        x=df_season["virtual_date"],
        y=df_season["ratio"],
        mode="lines",
        name=f"{season} - Ratio",
        line=dict(color=color_map[season], **line_styles["ratio"]),
        yaxis="y1",
        visible=True
    ))
    # SB
    fig_apr_jul.add_trace(go.Scatter(
        x=df_season["virtual_date"],
        y=df_season["SB"],
        mode="lines",
        name=f"{season} - SB",
        line=dict(color=color_map[season], **line_styles["SB"]),
        yaxis="y2",
        visible=True
    ))
    # CORN
    fig_apr_jul.add_trace(go.Scatter(
        x=df_season["virtual_date"],
        y=df_season["CORN"],
        mode="lines",
        name=f"{season} - CORN",
        line=dict(color=color_map[season], **line_styles["CORN"]),
        yaxis="y2",
        visible=True
    ))

# Create buttons to toggle each season
buttons = []
for i, season in enumerate(seasons):
    # Determine which traces belong to this season
    visible = [False] * len(fig_apr_jul.data)
    for j, trace in enumerate(fig_apr_jul.data):
        if trace.name.startswith(str(season)):
            visible[j] = True
    buttons.append(dict(
        label=str(season),
        method="update",
        args=[{"visible": visible},
              {"title": f"SB MAY / CORN APR (Showing {season})"}]
    ))
buttons.append(dict(
    label="Show All",
    method="update",
    args=[{"visible": [True]*len(fig_apr_jul.data)},  # all traces visible
          {"title": "SB MAY / CORN APR (All Seasons)"}]
))

# Layout
fig_apr_jul.update_layout(
    updatemenus=[dict(
        type="dropdown",
        active=0,
        buttons=buttons,
        x=0.95,
        y=1.15,
        xanchor="left",
        yanchor="top"
    )],
    title=f"SB MAY / CORN APR",
    template="plotly_white",
    height=700,
    width=1800,
    hovermode="x unified",
    xaxis=dict(tickformat="%b", dtick="M1", hoverformat="%d-%b"),
    yaxis=dict(title="Ratio", side="left"),
    yaxis2=dict(title="Prices (SB & CORN)", overlaying="y", side="right")
)

fig_apr_jul_html = fig_apr_jul.to_html(include_plotlyjs="cdn", full_html=True)




## WHEAT RATIOS

In [0]:
wheat_matba= zema.get_curve(curve="P-FUTURE-MATBA-INPUT-WHEAT-ROS-USD-MT", period=f"{start_date}::{end_date}")
wheat_matba=wheat_matba[wheat_matba['observation']=='Settle']

wheat_matba=wheat_matba[['date','value','contract_year','contract_month']]
wheat_matba['day']=wheat_matba['date'].dt.day
wheat_matba['month']=wheat_matba['date'].dt.month
wheat_matba['year']=wheat_matba['date'].dt.year
wheat_matba['virtual_date'] = pd.to_datetime(
    {'year': 2000, 'month': wheat_matba['month'], 'day': wheat_matba['day']},
    errors='coerce'  # This will convert invalid dates (e.g. Feb 30) to NaT
)
# Drop rows with invalid virtual dates
wheat_matba = wheat_matba.dropna(subset=['virtual_date'])

# You can now sort or use this date for seasonal charts
wheat_matba = wheat_matba.sort_values(by='virtual_date')
wheat_matba=wheat_matba[['date','value','contract_year','contract_month','virtual_date']]
wheat_matba['contract_date'] = (
    pd.to_datetime(dict(year=wheat_matba['contract_year'],
                        month=wheat_matba['contract_month'],
                        day=1))  # temporary first day of month
    + pd.offsets.MonthEnd(0)   # shift to last day of that month
)


### DEC WHEAT DEC CORN

In [0]:
dec_w = wheat_matba[wheat_matba["contract_month"] == 12].copy()
dec_c= corn_matba[corn_matba["contract_month"] == 12].copy()

dec_w_match = (dec_w.groupby("date")
                .apply(lambda x: x.loc[x["contract_date"] >= x.name].sort_values("contract_date").head(1))
                .reset_index(drop=True))

dec_c_match = (dec_c.groupby("date")
                .apply(lambda x: x.loc[x["contract_date"] >= x.name].sort_values("contract_date").head(1))
                .reset_index(drop=True))

dec_c_match.drop(["contract_year", "contract_month", "contract_date","virtual_date"], axis=1, inplace=True)
dec_c_match.rename(columns={"value": "CORN"}, inplace=True)


dec_w_match.drop(["contract_year", "contract_month", "contract_date","virtual_date"], axis=1, inplace=True)
dec_w_match.rename(columns={"value": "WHEAT"}, inplace=True)


# Merge dec_w + Jul
ratio_w_c = pd.merge(dec_c_match,dec_w_match, on="date")

ratio_w_c = ratio_w_c[ratio_w_c['CORN'] != 0]
ratio_w_c = ratio_w_c[ratio_w_c['WHEAT'] != 0]

# Compute spread
ratio_w_c["ratio"] = ratio_w_c["WHEAT"]/ratio_w_c["CORN"]
ratio_w_c['day']=ratio_w_c['date'].dt.day
ratio_w_c['month']=ratio_w_c['date'].dt.month
ratio_w_c['year']=ratio_w_c['date'].dt.year
ratio_w_c['virtual_date'] = pd.to_datetime(
    {'year': 2000, 'month': ratio_w_c['month'], 'day': ratio_w_c['day']},
    errors='coerce'  # This will convert invalid dates (e.g. Feb 30) to NaT
)


ratio_w_c['season'] = ratio_w_c['year']

# Drop rows with invalid virtual dates
ratio_w_c = ratio_w_c.dropna(subset=['virtual_date'])

# Create the base plot
fig_ratio_w_c = px.line(
    ratio_w_c,
    x="virtual_date",
    y="ratio",
    color="season",
    labels={
        "virtual_date": "",
        "ratio": "MATBA Ratios DEC WHEAT/DEC CORN ",
        "season": "season"
    },
    title="DEC WHEAT/DEC CORN",
)

# Compute average across seasons
avg_df = ratio_w_c.groupby("virtual_date", as_index=False)["ratio"].mean()
fig_ratio_w_c.add_scatter(
    x=avg_df["virtual_date"],
    y=avg_df["ratio"],
    mode="lines",
    name="Average",
    line=dict(color="gray", width=3, dash="dash")  # dashed gray line
)

# Identify last two seasons
seasons_sorted = sorted(ratio_w_c["season"].unique())
last_season = seasons_sorted[-1]
prev_season = seasons_sorted[-2]

# Update traces: highlight last and previous seasons
for trace in fig_ratio_w_c.data:
    if trace.name == str(last_season):
        trace.line.color = "red"
        trace.line.dash = "dash"
        trace.line.width = 5
    elif trace.name == str(prev_season):
        trace.line.color = "black"
        trace.line.dash = "solid"
        trace.line.width = 3

# Layout and hover
fig_ratio_w_c.update_traces(
    hovertemplate="%{x|%d-%b}<br>%{y:.2f}<extra>%{fullData.name}</extra>"
)
fig_ratio_w_c.update_layout(
    xaxis=dict(
        tickformat="%b",
        dtick="M1",
        hoverformat="%d-%b"
    ),
    yaxis_title="Price",
    template="plotly_white",
    height=700,
    width=1800,
    hovermode="x unified"
)



fig_ratio_w_c_html = fig_ratio_w_c.to_html(include_plotlyjs="cdn", full_html=True)

seasons = ratio_w_c["season"].unique()
line_styles = {
    "ratio": dict(dash="solid", width=2),
    "WHEAT": dict(dash="dot", width=2),
    "CORN": dict(dash="dash", width=2)
}
color_map = {season: px.colors.qualitative.Plotly[i % len(px.colors.qualitative.Plotly)]
             for i, season in enumerate(seasons)}

fig_wheat_corn = go.Figure()

# Add all traces
for season in seasons:
    df_season = ratio_w_c[ratio_w_c["season"] == season]
    # ratio
    fig_wheat_corn.add_trace(go.Scatter(
        x=df_season["virtual_date"],
        y=df_season["ratio"],
        mode="lines",
        name=f"{season} - Ratio",
        line=dict(color=color_map[season], **line_styles["ratio"]),
        yaxis="y1",
        visible=True
    ))
    # WHEAT
    fig_wheat_corn.add_trace(go.Scatter(
        x=df_season["virtual_date"],
        y=df_season["WHEAT"],
        mode="lines",
        name=f"{season} - WHEAT",
        line=dict(color=color_map[season], **line_styles["WHEAT"]),
        yaxis="y2",
        visible=True
    ))
    # CORN
    fig_wheat_corn.add_trace(go.Scatter(
        x=df_season["virtual_date"],
        y=df_season["CORN"],
        mode="lines",
        name=f"{season} - CORN",
        line=dict(color=color_map[season], **line_styles["CORN"]),
        yaxis="y2",
        visible=True
    ))

# Create buttons to toggle each season
buttons = []
for i, season in enumerate(seasons):
    # Determine which traces belong to this season
    visible = [False] * len(fig_wheat_corn.data)
    for j, trace in enumerate(fig_wheat_corn.data):
        if trace.name.startswith(str(season)):
            visible[j] = True
    buttons.append(dict(
        label=str(season),
        method="update",
        args=[{"visible": visible},
              {"title": f"MATBA WHEAT DEC/ CORN DEC (Showing {season})"}]
    ))
buttons.append(dict(
    label="Show All",
    method="update",
    args=[{"visible": [True]*len(fig_wheat_corn.data)},  # all traces visible
          {"title": "WHEAT DEC / CORN DEC (All Seasons)"}]
))

# Layout
fig_wheat_corn.update_layout(
    updatemenus=[dict(
        type="dropdown",
        active=0,
        buttons=buttons,
        x=0.95,
        y=1.15,
        xanchor="left",
        yanchor="top"
    )],
    title=f"WHEAT DEC / CORN DEC",
    template="plotly_white",
    height=700,
    width=1800,
    hovermode="x unified",
    xaxis=dict(tickformat="%b", dtick="M1", hoverformat="%d-%b"),
    yaxis=dict(title="Ratio", side="left"),
    yaxis2=dict(title="Prices (WHEAT & CORN)", overlaying="y", side="right")
)


fig_wheat_corn_html = fig_wheat_corn.to_html(include_plotlyjs="cdn", full_html=True)





## DEC WHEAT / NOV SB

In [0]:
dec_w = wheat_matba[wheat_matba["contract_month"] == 12].copy()
dec_sb= soy_matba[soy_matba["contract_month"] == 11].copy()

dec_w_match = (dec_w.groupby("date")
                .apply(lambda x: x.loc[x["contract_date"] >= x.name].sort_values("contract_date").head(1))
                .reset_index(drop=True))

dec_sb_match = (dec_sb.groupby("date")
                .apply(lambda x: x.loc[x["contract_date"] >= x.name].sort_values("contract_date").head(1))
                .reset_index(drop=True))

dec_sb_match.drop(["contract_year", "contract_month", "contract_date","virtual_date"], axis=1, inplace=True)
dec_sb_match.rename(columns={"value": "SB"}, inplace=True)


dec_w_match.drop(["contract_year", "contract_month", "contract_date","virtual_date"], axis=1, inplace=True)
dec_w_match.rename(columns={"value": "WHEAT"}, inplace=True)


# Merge dec_w + Jul
ratio_w_sb = pd.merge(dec_sb_match,dec_w_match, on="date")

ratio_w_sb = ratio_w_sb[ratio_w_sb['SB'] != 0]
ratio_w_sb = ratio_w_sb[ratio_w_sb['WHEAT'] != 0]

# Compute spread
ratio_w_sb["ratio"] = ratio_w_sb["WHEAT"]/ratio_w_sb["SB"]
ratio_w_sb['day']=ratio_w_sb['date'].dt.day
ratio_w_sb['month']=ratio_w_sb['date'].dt.month
ratio_w_sb['year']=ratio_w_sb['date'].dt.year
ratio_w_sb['virtual_date'] = pd.to_datetime(
    {'year': 2000, 'month': ratio_w_sb['month'], 'day': ratio_w_sb['day']},
    errors='coerce'  # This will convert invalid dates (e.g. Feb 30) to NaT
)

ratio_w_sb = ratio_w_sb[(ratio_w_sb['month'] <= 11)]

ratio_w_sb['season'] = ratio_w_sb['year']

# Drop rows with invalid virtual dates
ratio_w_sb = ratio_w_sb.dropna(subset=['virtual_date'])
ratio_w_sb

# Create the base plot
fig_ratio_w_sb = px.line(
    ratio_w_sb,
    x="virtual_date",
    y="ratio",
    color="season",
    labels={
        "virtual_date": "",
        "ratio": "MATBA Ratios DEC WHEAT/DEC SB ",
        "season": "season"
    },
    title="DEC WHEAT/DEC SB",
)

# Compute average across seasons
avg_df = ratio_w_sb.groupby("virtual_date", as_index=False)["ratio"].mean()
fig_ratio_w_sb.add_scatter(
    x=avg_df["virtual_date"],
    y=avg_df["ratio"],
    mode="lines",
    name="Average",
    line=dict(color="gray", width=3, dash="dash")  # dashed gray line
)

# Identify last two seasons
seasons_sorted = sorted(ratio_w_sb["season"].unique())
last_season = seasons_sorted[-1]
prev_season = seasons_sorted[-2]

# Update traces: highlight last and previous seasons
for trace in fig_ratio_w_sb.data:
    if trace.name == str(last_season):
        trace.line.color = "red"
        trace.line.dash = "dash"
        trace.line.width = 5
    elif trace.name == str(prev_season):
        trace.line.color = "black"
        trace.line.dash = "solid"
        trace.line.width = 3

# Layout and hover
fig_ratio_w_sb.update_traces(
    hovertemplate="%{x|%d-%b}<br>%{y:.2f}<extra>%{fullData.name}</extra>"
)
fig_ratio_w_sb.update_layout(
    xaxis=dict(
        tickformat="%b",
        dtick="M1",
        hoverformat="%d-%b"
    ),
    yaxis_title="Price",
    template="plotly_white",
    height=700,
    width=1800,
    hovermode="x unified"
)


fig_ratio_w_sb_html = fig_ratio_w_sb.to_html(include_plotlyjs="cdn", full_html=True)


seasons = ratio_w_sb["season"].unique()
line_styles = {
    "ratio": dict(dash="solid", width=2),
    "WHEAT": dict(dash="dot", width=2),
    "SB": dict(dash="dash", width=2)
}
color_map = {season: px.colors.qualitative.Plotly[i % len(px.colors.qualitative.Plotly)]
             for i, season in enumerate(seasons)}

fig_wheat_sb = go.Figure()

# Add all traces
for season in seasons:
    df_season = ratio_w_sb[ratio_w_sb["season"] == season]
    # ratio
    fig_wheat_sb.add_trace(go.Scatter(
        x=df_season["virtual_date"],
        y=df_season["ratio"],
        mode="lines",
        name=f"{season} - Ratio",
        line=dict(color=color_map[season], **line_styles["ratio"]),
        yaxis="y1",
        visible=True
    ))
    # WHEAT
    fig_wheat_sb.add_trace(go.Scatter(
        x=df_season["virtual_date"],
        y=df_season["WHEAT"],
        mode="lines",
        name=f"{season} - WHEAT",
        line=dict(color=color_map[season], **line_styles["WHEAT"]),
        yaxis="y2",
        visible=True
    ))
    # CORN
    fig_wheat_sb.add_trace(go.Scatter(
        x=df_season["virtual_date"],
        y=df_season["SB"],
        mode="lines",
        name=f"{season} - SB",
        line=dict(color=color_map[season], **line_styles["SB"]),
        yaxis="y2",
        visible=True
    ))

# Create buttons to toggle each season
buttons = []
for i, season in enumerate(seasons):
    # Determine which traces belong to this season
    visible = [False] * len(fig_wheat_sb.data)
    for j, trace in enumerate(fig_wheat_sb.data):
        if trace.name.startswith(str(season)):
            visible[j] = True
    buttons.append(dict(
        label=str(season),
        method="update",
        args=[{"visible": visible},
              {"title": f"MATBA WHEAT DEC/ SB DEC (Showing {season})"}]
    ))
buttons.append(dict(
    label="Show All",
    method="update",
    args=[{"visible": [True]*len(fig_wheat_sb.data)},  # all traces visible
          {"title": "WHEAT DEC / SB DEC (All Seasons)"}]
))

# Layout
fig_wheat_sb.update_layout(
    updatemenus=[dict(
        type="dropdown",
        active=0,
        buttons=buttons,
        x=0.95,
        y=1.15,
        xanchor="left",
        yanchor="top"
    )],
    title=f"WHEAT DEC / SB DEC",
    template="plotly_white",
    height=700,
    width=1800,
    hovermode="x unified",
    xaxis=dict(tickformat="%b", dtick="M1", hoverformat="%d-%b"),
    yaxis=dict(title="Ratio", side="left"),
    yaxis2=dict(title="Prices (WHEAT & SB)", overlaying="y", side="right")
)


fig_wheat_sb_html = fig_wheat_sb.to_html(include_plotlyjs="cdn", full_html=True)




# MAIL

In [0]:
wheat_html_report  = f"""
<!DOCTYPE html>
<html>
<head>
    <title>Markets Snapshot</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            padding: 20px;
        }}
        h2 {{
            margin-top: 40px;
            color: #2c3e50;
        }}
        .chart-container {{
            margin-bottom: 50px;
        }}
    </style>
</head>
<body>
    <h1>Wheat Contracts</h1>

    <h2>SPOT</h2>
    <div class="chart-container">{spot_html}</div>

    <h2>SPOT VS MATIF</h2>
    <div class="chart-container">{spot_matif_html}</div>

    <h2>SPOT VS HRW</h2>
    <div class="chart-container">{spot_hrw_html}</div>

    <h2>SPOT VS Reference contract (MATIF: 1stNov-Feb-12th - Rest: HRW)</h2>
    <div class="chart-container">{spot_ref_html}</div>

    <h2>December FOB Upr VS MATIF</h2>
    <div class="chart-container">{DEC_arg_wheat_upbb_html}</div>
</body>
</html>
"""

corn_html_report  = f"""
<!DOCTYPE html>
<html>
<head>
    <title>Markets Snapshot</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            padding: 20px;
        }}
        h2 {{
            margin-top: 40px;
            color: #2c3e50;
        }}
        .chart-container {{
            margin-bottom: 50px;
        }}
    </style>
</head>
<body>
    <h1>Corn Contracts</h1>

        <h2>SPOT</h2>
        <div class="chart-container">{spot_corn_html}</div>
        
        <h2>Spot BASIS Corn</h2>
        <div class="chart-container">{prem_spot_corn_html}</div>

        <h2>UPR FOB AND PREMIUM</h2>
        <h2>APRIL FOB AND PREM</h2>
        <div class="chart-container">{APR_arg_corn_upbb_html}</div>
        
        <h2>JUL FOB AND PREM</h2>
        <div class="chart-container">{JUL_arg_corn_upbb_html}</div>

        <h2>UPR VS CBOT</h2>
        <div class="chart-container">{cbot_vs_upr_html}</div>

        <h2>Spot BB VS UPR</h2>
        <div class="chart-container">{corn_bb_upr_prem_html}</div>

        <h2>MATBA</h2>
        <h3>MATBA APR</h3>
        <div class="chart-container">{apr_matba_html}</div>
        <h3>MATBA JUL</h3>
        <div class="chart-container">{JUL_matba_html}</div>
        <h3>MATBA DEC</h3>
        <div class="chart-container">{dec_matba_html}</div>
        <h3>MATBA SPREAD JUL - APR</h3>
        <div class="chart-container">{spread_apr_july_html}</div>
        <h3>MATBA SPREAD DEC - JUL</h3>
        <div class="chart-container">{spread_july_dec_html}</div>

        <h2>July</h2>
        <div class="chart-container">{jul_prem_corn_html}</div>





</body>
</html>
"""

barley_html_report  = f"""
<!DOCTYPE html>
<html>
<head>
    <title>Markets Snapshot</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            padding: 20px;
        }}
        h2 {{
            margin-top: 40px;
            color: #2c3e50;
        }}
        .chart-container {{
            margin-bottom: 50px;
        }}
    </style>
</head>
<body>
    <h1>MARCH MATIF VS ARG FOB BARLEY</h1>

    <div class="chart-container">{prem_barley_matif_html}</div>

    <h1>DEC MATIF VS ARG FOB BARLEY</h1>

    <div class="chart-container">{prem_barley_matif_dec_html}</div>

    <h1>WHEAT UPR VS ARG FOB BARLEY</h1>

    <div class="chart-container">{upr_w_vs_barley_html}</div>

    <h1>CORN BB VS ARG FOB BARLEY</h1>

    <div class="chart-container">{BB_c_vs_barley_html}</div>


</body>
</html>
"""

ratio_html  = f"""
<!DOCTYPE html>
<html>
<head>
    <title>MATBA SB/CORN</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            padding: 20px;
        }}
        h2 {{
            margin-top: 40px;
            color: #2c3e50;
        }}
        .chart-container {{
            margin-bottom: 50px;
        }}
    </style>
</head>
<body>
    <h1>MATBA RATIO SB/CORN</h1>
    <h2>Ratio</h2>
    <div class="chart-container">{ratio_apr_july_html}</div>
    <h2>Ratio and Prices</h2>
    <div class="chart-container">{fig_apr_jul_html}</div>
    

    <h1>MATBA RATIO DEC WHEAT/ DEC CORN</h1>
    <h2>Ratio</h2>
    <div class="chart-container">{fig_ratio_w_c_html}</div>
    <h2>Ratio and Prices</h2>
    <div class="chart-container">{fig_wheat_corn_html}</div>

    <h1>MATBA RATIO DEC WHEAT/ DEC SB</h1>
    <h2>Ratio</h2>
    <div class="chart-container">{fig_ratio_w_sb_html}</div>
    <h2>Ratio and Prices</h2>
    <div class="chart-container">{fig_wheat_sb_html}</div>


</body>
</html>
"""
ratio_html_report_bytes = ratio_html.encode("utf-8")

wheat_report_bytes = wheat_html_report.encode("utf-8")
corn_report_bytes = corn_html_report.encode("utf-8")
barley_report_bytes = barley_html_report.encode("utf-8")

In [0]:
html_content = f"""
<!DOCTYPE html>
<html>
<head>
    <title>Markets Snapshot</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            margin: 20px;
        }}
        h1, h2 {{
            margin-top: 40px;
            color: #333;
        }}
        img {{
            width: 1800px;
            max-width: 100%;
            min-width: 1000px;
            display: block;
            margin-bottom: 30px;
        }}
    </style>
</head>
<body>
    <h1>Wheat Contracts</h1>

    <h2>SPOT</h2>
    <img src="spot_wheat" alt="Spot">

    <h2>SPOT VS MATIF</h2>
    <img src="w_vs_matif" alt="SPOT vs MATIF">

    <h2>SPOT VS HRW</h2>
    <img src="spot_hrw" alt="SPOT vs HRW">

    <h2>SPOT VS Reference contract (MATIF: 1stNov-Feb-12th - Rest: HRW)</h2>
    <img src="spot_ref" alt="SPOT vs REF">

    <h2>December FOB UPR VS MATIF</h2>
    <img src="w_dec" alt="December Wheat">

    <h1>Corn Contracts</h1>

    <h2>SPOT</h2>
    <img src="c_spot" alt="Corn Spot">

    <h2>Spot BASIS Corn</h2>
    <img src="prem_corn" alt="prem_spot_corn">

    <h2>CBOT VS UPR</h2>
    <img src="cbot_vs_upr" alt="prem_spot_corn">

    <h2>BB VS UPR</h2>
    <img src="corn_bb_upr_prem" alt="prem_spot_corn">
    
    <h2>DEC MATBA</h2>
    <img src="c_dec" alt="December Corn">

    <h1 style="margin-top: 20px;">BARLEY</h1>

    <h2 style="margin-top: 20px;"> MATIF VS FOB BARLEY </h1>
    <img src="matif_barley" alt="Spot Chart"
         style="display: block; margin: 20px auto; width: 2600px; height: auto;">

    <h2 style="margin-top: 20px;">DEC MATIF VS FOB BARLEY</h1>
    <img src="dec_matif_barley" alt="Spot Chart"
         style="display: block; margin: 20px auto; width: 2600px; height: auto;">

    <h2 style="margin-top: 20px;">UPR WHEAT VS FOB BARLEY</h1>
    <img src="w_upr_barley" alt="Spot Chart"
         style="display: block; margin: 20px auto; width: 2600px; height: auto;">

    <h2 style="margin-top: 20px;">BB CORN VS FOB BARLEY</h1>
    <img src="corn_bb_barley" alt="Spot Chart"
         style="display: block; margin: 20px auto; width: 2600px; height: auto;">

</body>
</html>
"""

grains=['florian.girardi-ext@ldc.com','Roman.Avramishin@LDC.com','juan.garciafuentes@ldc.com','gonzalo.lascombes@ldc.com','juan.carnemolla@LDC.com','valentin.chiesa@ldc.com']
test=['florian.girardi-ext@ldc.com']

# # Send email with embedded chart and table
LDCDataAccessLayerPy.mail.mail_send(
    to=test,
    subject=f'Prices and ratios daily report {datetime.now().strftime("%d-%m")}',
    from_addr="florian.girardi-ext@ldc.com",
    body="Please find attached the report",
    mime_type="html",
    html_images={"spot_wheat": spot,"w_vs_matif": spot_matif,"spot_ref": spot_ref,"spot_hrw":spot_hrw,"chart6": spot_corn,"w_dec": DEC_arg_wheat_upbb,"c_spot": spot_corn,"corn_bb_upr_prem":corn_bb_upr_prem,"prem_corn": prem_spot_corn,"cbot_vs_upr":cbot_vs_upr, "c_apr":apr_prem_corn, 'c_jul':jul_prem_corn,'c_dec':dec_matba,"matif_barley":prem_barley_matif,"dec_matif_barley":prem_barley_matif_dec,"w_upr_barley":upr_w_vs_barley,"corn_bb_barley":BB_c_vs_barley},
    attachment={"wheat_report.html": wheat_report_bytes,"corn_report.html": corn_report_bytes,"barley_report.html":barley_report_bytes,"MATBA_ratio.html":ratio_html_report_bytes}
)



In [0]:
html_content_ratio = f"""
<!DOCTYPE html>
<html>
<head>
    <title>MATBA RATIOS</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            margin: 20px;
        }}
        h1, h2 {{
            margin-top: 40px;
            color: #333;
        }}
        img {{
            width: 1800px;
            max-width: 100%;
            min-width: 1000px;
            display: block;
            margin-bottom: 30px;
        }}
    </style>
</head>
<body>
    <h1>MATBA RATIO SB/CORN</h1>
    <img src="ratio_apr_july" alt="SB/CORN">

    <h1>MATBA RATIO DEC WHEAT/ DEC CORN</h1>
    <img src="fig_ratio_w_c" alt="DEC WHEAT/CORN">

    <h1>MATBA RATIO DEC WHEAT/ DEC SB</h1>
    <img src="fig_ratio_w_sb" alt="DEC WHEAT/SB">

</body>
</html>
"""

In [0]:
      
# beans=['florian.girardi-ext@ldc.com','mateo.vergniaud@ldc.com','lucas.bel@ldc.com']
# # Send email with embedded chart and table
# LDCDataAccessLayerPy.mail.mail_send(
#     to=beans,
#     subject=f'MATBA Ratios {datetime.today().strftime("%d-%m")}',
#     from_addr="florian.girardi-ext@ldc.com",
#     mime_type="html",
#     body=html_content_ratio, 
#     html_images={"ratio_apr_july": ratio_apr_july,"fig_ratio_w_c": fig_ratio_w_c,"fig_ratio_w_sb": fig_ratio_w_sb},
#     attachment={"matba_ratios.html": ratio_html_report_bytes}
# )